# 1. Load Data

a) Load Data and drop irrelevant Features

In [ ]:
import pandas as pd

# Load data
data = pd.read_excel("ML_DATA SLE RELAPSE SSM.xlsx")

# Drop unwanted columns
excluded_features = [
    'SUBJECT ID',
    'DATE_OF_CLINIC_VISIT_W_RELAPSE_OR_LAST_TCA',
    'YEAR BIRTH',
    'Unnamed: 16',
    'Unnamed: 26',
    'RELAPSE1,NONRELAPSE0',
    'MMF1,AZA2,3 MTX',
    'REMISSION INTERVAL',
    'AGE RELAPSE'
]
data = data.drop(columns=excluded_features)

# Rename 'Unnamed: 2' → 'RELAPSE1,NONRELAPSE0'
data = data.rename(columns={'Unnamed: 2': 'RELAPSE1,NONRELAPSE0'})

In [ ]:
data.head(5)


b) Target Feature Distribution Check

In [ ]:
# Check ratio of classes for RELAPSE1,NONRELAPSE0
print("RELAPSE1,NONRELAPSE0 class distribution:")
print(data['RELAPSE1,NONRELAPSE0'].value_counts(normalize=True))  # normalize=True gives ratio

c) Display a summary of dataset:


In [ ]:
# Get the dimensions of the dataset
num_rows, num_columns = data.shape

print(f"The dataset has {num_rows} rows and {num_columns} columns.")

# 2.  Data Preprocessing

**a. Feature Type Splitting**

In [ ]:
def get_feature_columns():
    """
    Returns lists of numerical and categorical feature columns
    for the SLE Relapse project dataset.
    """

    # Target column
    target_column = 'RELAPSE1,NONRELAPSE0'

    # Numerical Features (continuous labs, ages, counts)
    numeric_columns = [
        'AGE',
        'AGE SLE DIAGNOSIS',
        'AGE RELAPSE',
        'C3 LEVEL',
        'C4 LEVEL',
        'LOWEST HB',
        'LOWEST WCC',
        'LOWEST NEUT',
        'LOWEST LYMP',
        'LOWEST PLATELT',
        'LOWEST ALB',
        'HIGHEST CRP',
        'HIGHEST ESR',
        'REMISSION INTERVAL',
        'LOWEST PRED',        # ordinal, small range (0–12)
        'DISEASE DURATION'
    ]

    # Categorical Features (binary or low-cardinality discrete)
    categorical_columns = [
        'FEMALE0 MALE1',
        'MALAY0, CHINESE1,INDIAN2',
        'ANTIDSDNA12M_POSITIVE1',
        'C3_12M_LOW1, NORMAL0',
        '12M_ C4 LOW1,NORMAL0',
        'RENAL1 NON-RENAL0',
        'INFECTION YES1',
        'IMMUNOSUPPRESSANT',
        'HCQ'
    ]

    return target_column, numeric_columns, categorical_columns


**b. Train/Test Split using stratified split**

In [ ]:
from sklearn.model_selection import train_test_split

def stratified_split(df, target_column, test_size=0.2, random_state=42):


    # Separate features and target
    X = df.drop(columns=[target_column])
    y = df[target_column]

    # Stratified split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state
    )

    return X_train, X_test, y_train, y_test


**c. Missing Data Imputation**

In [ ]:
def fit_imputation_values(train, numeric_columns, categorical_columns, high_missing_threshold=0.4):
    """Learn all imputation values and missingness flags from training data only."""
    state = {
        "fill_values": {},
        "missing_indicator_columns": []
    }

    for col in numeric_columns:
        if col == "LOWEST PRED":
            state["fill_values"][col] = 0
        else:
            state["fill_values"][col] = train[col].median()

        if (
            col != "LOWEST PRED"
            and train[col].isna().mean() > high_missing_threshold
        ):
            state["missing_indicator_columns"].append(col)

    for col in categorical_columns:
        observed = train[col].dropna()
        unique_values = set(observed.unique())

        if not observed.empty and unique_values.issubset({0, 1}):
            state["fill_values"][col] = 2
        elif not observed.empty:
            state["fill_values"][col] = observed.mode().iloc[0]
        else:
            raise ValueError(f"Cannot fit imputation for entirely missing column: {col}")

    return state


def apply_imputation(df, imputation_state):
    """Apply training-derived imputation values to another dataframe."""
    result = df.copy()

    for col in imputation_state["missing_indicator_columns"]:
        result[f"{col}_MISSING"] = result[col].isna().astype(int)

    for col, fill_value in imputation_state["fill_values"].items():
        result[col] = result[col].fillna(fill_value)

    return result


**d. Feature engineering**

In [ ]:
def engineer_lowest_pred(df):

    df = df.copy()

    # Overwrite 'LOWEST PRED' with binned values
    df["LOWEST PRED"] = pd.cut(
        df["LOWEST PRED"],
        bins=[-1, 5, 10],
        labels=["0", "1"]
    ).astype(str)

    # Rename column to new name (index stays the same)
    df = df.rename(columns={"LOWEST PRED": "LOWEST_PRED_BIN"})

    return df


**e. outlier handling (transformation)**
  - box cox: LOWEST HB, LOWEST ALB
  - log: HIGHEST CRP, LOWEST WCC, LOWEST NEUT

In [ ]:
from scipy.stats import boxcox
import numpy as np

def apply_boxcox(train, test, features):
    """Fit Box-Cox parameters on training data and apply them to test data."""
    train = train.copy()
    test = test.copy()
    lambdas = {}

    for col in features:
        train_min = train[col].min()
        shift = abs(train_min) + 1e-6 if train_min <= 0 else 0

        train_values = train[col] + shift
        test_values = test[col] + shift

        if (test_values <= 0).any():
            raise ValueError(
                f"Training-derived Box-Cox shift leaves non-positive test values in {col}"
            )

        transformed_train, fitted_lambda = boxcox(train_values)
        train[col] = transformed_train
        test[col] = boxcox(test_values, lmbda=fitted_lambda)
        lambdas[col] = {"lambda": fitted_lambda, "shift": shift}

    return train, test, lambdas


In [ ]:
def apply_log_transform(train, test, features):
    """Derive log shifts from training data and reuse them for test data."""
    train = train.copy()
    test = test.copy()

    for col in features:
        train_min = train[col].min()
        shift = abs(train_min) + 1e-6 if train_min <= 0 else 0

        train_values = train[col] + shift
        test_values = test[col] + shift

        if (test_values <= 0).any():
            raise ValueError(
                f"Training-derived log shift leaves non-positive test values in {col}"
            )

        train[col] = np.log(train_values)
        test[col] = np.log(test_values)

    return train, test


scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler
import pandas as pd

def minmax_scale(train_df: pd.DataFrame, test_df: pd.DataFrame):

    scaler = MinMaxScaler()

    # Identify numeric columns
    num_cols = train_df.select_dtypes(include=['int64','float64']).columns

    # Fit scaler on train, transform both train & test
    train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
    test_df[num_cols] = scaler.transform(test_df[num_cols])

    return train_df, test_df


**f. Implement Pipeline**

CatBoost

In [ ]:
# === 3. Get Feature Columns ===
target_col, numeric_cols, categorical_cols = get_feature_columns()

# Keep only features still present after excluded_features are removed.
numeric_cols = [col for col in numeric_cols if col in data.columns]
categorical_cols = [col for col in categorical_cols if col in data.columns]

# === 4. Stratified Train/Test Split (before learned preprocessing) ===
X_train_raw, X_test_raw, y_train, y_test = stratified_split(data, target_col)

# === 5. Fit imputation on training data only, then apply to both sets ===
imputation_state = fit_imputation_values(
    X_train_raw,
    numeric_cols,
    categorical_cols
)
X_train_imputed = apply_imputation(X_train_raw, imputation_state)
X_test_imputed = apply_imputation(X_test_raw, imputation_state)

# === 6. Shared engineered/transformed base ===
X_train_base = engineer_lowest_pred(X_train_imputed)
X_test_base = engineer_lowest_pred(X_test_imputed)

boxcox_features = ["LOWEST HB", "LOWEST ALB"]
X_train_base, X_test_base, lambdas = apply_boxcox(
    X_train_base,
    X_test_base,
    boxcox_features
)

log_features = ["HIGHEST CRP", "LOWEST WCC", "LOWEST NEUT"]
X_train_base, X_test_base = apply_log_transform(
    X_train_base,
    X_test_base,
    log_features
)

# === 7. CatBoost-ready branch: categorical columns preserved ===
X_train_cb = X_train_base.copy()
X_test_cb = X_test_base.copy()

model_categorical_cols = categorical_cols + ["LOWEST_PRED_BIN"]
cat_features_cb = [
    col for col in model_categorical_cols
    if col in X_train_cb.columns
]

for col in cat_features_cb:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

# Compatibility aliases for the existing CatBoost modelling cells.
X_train = X_train_cb
X_test = X_test_cb


Other Tree & SKlearn models

In [ ]:
# === 8. Tree/EBM-ready branch: encoded and unscaled ===
X_train_tree = pd.get_dummies(
    X_train_base.copy(),
    columns=model_categorical_cols,
    drop_first=True,
    dtype=int
)
X_test_tree = pd.get_dummies(
    X_test_base.copy(),
    columns=model_categorical_cols,
    drop_first=True,
    dtype=int
)

X_train_tree, X_test_tree = X_train_tree.align(
    X_test_tree,
    join="left",
    axis=1,
    fill_value=0
)

# === 9. Scaled sklearn branch: derived from the unscaled tree branch ===
from sklearn.preprocessing import MinMaxScaler

X_train_ml = X_train_tree.copy()
X_test_ml = X_test_tree.copy()
numeric_cols_for_scaling_ml = X_train_ml.select_dtypes(
    include=["number"]
).columns.tolist()

scaler_ml = MinMaxScaler()
X_train_ml[numeric_cols_for_scaling_ml] = scaler_ml.fit_transform(
    X_train_ml[numeric_cols_for_scaling_ml]
)
X_test_ml[numeric_cols_for_scaling_ml] = scaler_ml.transform(
    X_test_ml[numeric_cols_for_scaling_ml]
)

# Compatibility aliases for the existing Logistic Regression cells.
y_train_lr = y_train.copy()
y_test_lr = y_test.copy()
X_train_lr = X_train_ml
X_test_lr = X_test_ml


Sanity Check

# 3. Feature Selection and Model Benchmarking

Feature selection is performed on the training data only. Each model uses the preprocessing branch appropriate to its learning algorithm.

## 3.1 Shared imports and feature-name mapping

LR/RF selections use one-hot encoded column names, while CatBoost uses original clinical feature names. The helper below maps encoded columns back to their original clinical feature group for consensus voting, then expands consensus groups back to all corresponding encoded columns.

In [ ]:
from collections import Counter
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from boruta import BorutaPy
from catboost import CatBoostClassifier

# Compatibility shims for the currently published BorutaShap package with
# NumPy >= 2.0 and SciPy versions where scipy.stats.binom_test was removed.
if not hasattr(np, "NaN"):
    np.NaN = np.nan

import scipy.stats as scipy_stats
if not hasattr(scipy_stats, "binom_test"):
    def _legacy_binom_test(x, n=None, p=0.5, alternative="two-sided"):
        return scipy_stats.binomtest(
            k=int(x),
            n=int(n),
            p=p,
            alternative=alternative
        ).pvalue
    scipy_stats.binom_test = _legacy_binom_test

from BorutaShap import BorutaShap

RANDOM_STATE = 42
original_feature_names = list(X_train_cb.columns)
original_feature_names_by_length = sorted(
    original_feature_names,
    key=len,
    reverse=True
)


def encoded_to_original_feature(encoded_feature):
    """Map an encoded column to its original CatBoost-ready feature name."""
    if encoded_feature in original_feature_names:
        return encoded_feature

    for original_feature in original_feature_names_by_length:
        if encoded_feature.startswith(f"{original_feature}_"):
            return original_feature

    raise KeyError(f"No original feature mapping found for: {encoded_feature}")


def expand_original_groups_to_encoded(original_groups, encoded_columns):
    """Include every one-hot column belonging to a selected original feature."""
    selected_groups = set(original_groups)
    return [
        column for column in encoded_columns
        if encoded_to_original_feature(column) in selected_groups
    ]


## 3.2 Logistic Regression feature selection ? L1 regularization

In [ ]:
lr_feature_selector = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=1.0,
    class_weight="balanced",
    max_iter=2000,
    random_state=RANDOM_STATE
)
lr_feature_selector.fit(X_train_ml, y_train)

lr_coefficients = np.abs(lr_feature_selector.coef_).ravel()
selected_features_lr = X_train_ml.columns[lr_coefficients > 1e-8].tolist()

if not selected_features_lr:
    fallback_index = int(np.argmax(lr_coefficients))
    selected_features_lr = [X_train_ml.columns[fallback_index]]
    warnings.warn(
        "L1 selected no non-zero coefficients; using the largest absolute coefficient."
    )

print("Selected LR features:")
print(selected_features_lr)


## 3.3 Random Forest feature selection ? Boruta

In [ ]:
rf_boruta_estimator = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_boruta_selector = BorutaPy(
    estimator=rf_boruta_estimator,
    n_estimators="auto",
    max_iter=50,
    random_state=RANDOM_STATE,
    verbose=0
)
rf_boruta_selector.fit(X_train_tree.to_numpy(), y_train.to_numpy())

selected_features_rf = X_train_tree.columns[
    rf_boruta_selector.support_
].tolist()

if not selected_features_rf:
    selected_features_rf = X_train_tree.columns[
        rf_boruta_selector.support_weak_
    ].tolist()
    warnings.warn(
        "Boruta confirmed no RF features; using its tentative features."
    )

if not selected_features_rf:
    best_rank = np.min(rf_boruta_selector.ranking_)
    selected_features_rf = X_train_tree.columns[
        rf_boruta_selector.ranking_ == best_rank
    ].tolist()
    warnings.warn(
        "Boruta returned no confirmed or tentative RF features; using its best-ranked features."
    )

print("Selected RF features:")
print(selected_features_rf)


## 3.4 CatBoost feature selection ? BorutaSHAP

In [ ]:
class_ratio = y_train.value_counts()[0] / y_train.value_counts()[1]

catboost_boruta_model = CatBoostClassifier(
    iterations=100,
    depth=4,
    learning_rate=0.05,
    loss_function="Logloss",
    class_weights=[1, class_ratio],
    random_seed=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False
)

# BorutaShap's categorical detection expects object/category dtypes.
X_train_cb_boruta = X_train_cb.copy()
for column in cat_features_cb:
    X_train_cb_boruta[column] = X_train_cb_boruta[column].astype(object)

catboost_boruta_selector = BorutaShap(
    model=catboost_boruta_model,
    importance_measure="shap",
    classification=True,
    percentile=100,
    pvalue=0.05
)
catboost_boruta_selector.fit(
    X=X_train_cb_boruta,
    y=y_train,
    n_trials=20,
    random_state=RANDOM_STATE,
    sample=False,
    train_or_test="train",
    normalize=True,
    verbose=False
)

selected_features_catboost = list(catboost_boruta_selector.accepted)

if len(selected_features_catboost) < 2 and catboost_boruta_selector.tentative:
    catboost_boruta_selector.TentativeRoughFix()
    selected_features_catboost = list(catboost_boruta_selector.accepted)

if len(selected_features_catboost) < 2:
    # Small-data fallback remains BorutaSHAP-derived: rank original features by
    # median importance history rather than reverting to the old SHAP top-k flow.
    median_borutashap_importance = (
        catboost_boruta_selector.history_x[original_feature_names]
        .median(axis=0)
        .sort_values(ascending=False)
    )
    selected_features_catboost = median_borutashap_importance.head(5).index.tolist()
    warnings.warn(
        "BorutaSHAP confirmed fewer than two features; using its five highest "
        "median-importance features for a trainable CatBoost benchmark."
    )

print("Selected CatBoost features:")
print(selected_features_catboost)


## 3.5 MLP 2-out-of-3 consensus feature set

In [ ]:
selected_feature_groups_lr = {
    encoded_to_original_feature(feature)
    for feature in selected_features_lr
}
selected_feature_groups_rf = {
    encoded_to_original_feature(feature)
    for feature in selected_features_rf
}
selected_feature_groups_catboost = set(selected_features_catboost)

feature_votes = Counter()
for selected_group_set in (
    selected_feature_groups_lr,
    selected_feature_groups_rf,
    selected_feature_groups_catboost
):
    feature_votes.update(selected_group_set)

consensus_feature_groups = [
    feature for feature in original_feature_names
    if feature_votes[feature] >= 2
]

MIN_MLP_FEATURE_GROUPS = 2
if len(consensus_feature_groups) < MIN_MLP_FEATURE_GROUPS:
    warnings.warn(
        "The 2-out-of-3 MLP consensus is empty or too small; using the union "
        "of LR, RF, and CatBoost selected feature groups."
    )
    union_groups = (
        selected_feature_groups_lr
        | selected_feature_groups_rf
        | selected_feature_groups_catboost
    )
    consensus_feature_groups = [
        feature for feature in original_feature_names
        if feature in union_groups
    ]

selected_features_mlp = expand_original_groups_to_encoded(
    consensus_feature_groups,
    X_train_ml.columns
)

if not selected_features_mlp:
    raise RuntimeError("No encoded MLP features remain after consensus mapping.")

print(f"LR selected features: {len(selected_features_lr)}")
print(f"RF selected features: {len(selected_features_rf)}")
print(f"CatBoost selected features: {len(selected_features_catboost)}")
print(f"MLP consensus features: {len(selected_features_mlp)}")
print("Selected MLP consensus features:")
print(selected_features_mlp)


## 3.6 Prepare model-specific selected datasets

In [ ]:
X_train_lr_sel = X_train_ml[selected_features_lr].copy()
X_test_lr_sel = X_test_ml[selected_features_lr].copy()

X_train_rf_sel = X_train_tree[selected_features_rf].copy()
X_test_rf_sel = X_test_tree[selected_features_rf].copy()

X_train_mlp_sel = X_train_ml[selected_features_mlp].copy()
X_test_mlp_sel = X_test_ml[selected_features_mlp].copy()

X_train_cb_sel = X_train_cb[selected_features_catboost].copy()
X_test_cb_sel = X_test_cb[selected_features_catboost].copy()
cat_features_cb_sel = [
    feature for feature in selected_features_catboost
    if feature in cat_features_cb
]


## 3.7 Benchmark Logistic Regression, Random Forest, MLP, and CatBoost

In [ ]:
benchmark_models = {
    "Logistic Regression": LogisticRegression(
        penalty="l1",
        solver="liblinear",
        C=1.0,
        class_weight="balanced",
        max_iter=2000,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=1
    ),
    "MLP": MLPClassifier(
        hidden_layer_sizes=(16,),
        alpha=0.01,
        early_stopping=True,
        validation_fraction=0.2,
        max_iter=1000,
        random_state=RANDOM_STATE
    ),
    "CatBoost": CatBoostClassifier(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        loss_function="Logloss",
        class_weights=[1, class_ratio],
        random_seed=RANDOM_STATE,
        verbose=0,
        allow_writing_files=False
    )
}

benchmark_datasets = {
    "Logistic Regression": (X_train_lr_sel, X_test_lr_sel),
    "Random Forest": (X_train_rf_sel, X_test_rf_sel),
    "MLP": (X_train_mlp_sel, X_test_mlp_sel),
    "CatBoost": (X_train_cb_sel, X_test_cb_sel)
}

feature_selection_methods = {
    "Logistic Regression": "L1 Logistic Regression",
    "Random Forest": "Boruta",
    "MLP": "2-out-of-3 consensus",
    "CatBoost": "BorutaSHAP"
}

selected_features_by_model = {
    "Logistic Regression": selected_features_lr,
    "Random Forest": selected_features_rf,
    "MLP": selected_features_mlp,
    "CatBoost": selected_features_catboost
}

fitted_models = {}
confusion_matrices = {}
benchmark_rows = []

for model_name, model in benchmark_models.items():
    model_X_train, model_X_test = benchmark_datasets[model_name]

    if model_name == "CatBoost":
        model.fit(
            model_X_train,
            y_train,
            cat_features=cat_features_cb_sel
        )
    else:
        model.fit(model_X_train, y_train)

    predictions = np.asarray(model.predict(model_X_test)).ravel().astype(int)
    matrix = confusion_matrix(y_test, predictions)

    fitted_models[model_name] = model
    confusion_matrices[model_name] = matrix
    benchmark_rows.append({
        "Model": model_name,
        "Feature_Selection_Method": feature_selection_methods[model_name],
        "Num_Features": model_X_train.shape[1],
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "Confusion_Matrix": matrix.tolist()
    })

benchmark_results_df = (
    pd.DataFrame(benchmark_rows)
    .sort_values(["F1", "Recall", "Precision"], ascending=False)
    .reset_index(drop=True)
)

print("Benchmark results:")
print(benchmark_results_df.to_string(index=False))
print("\nConfusion matrices:")
for model_name, matrix in confusion_matrices.items():
    print(f"{model_name}:\n{matrix}")


## 3.8 Select the best benchmark model

In [ ]:
best_row = benchmark_results_df.iloc[0]
best_model_name = best_row["Model"]
best_model = fitted_models[best_model_name]
best_feature_set = selected_features_by_model[best_model_name]
best_X_train, best_X_test = benchmark_datasets[best_model_name]

print(f"Best model name: {best_model_name}")
print(f"Best model precision: {best_row['Precision']:.4f}")
print(f"Best model recall: {best_row['Recall']:.4f}")
print(f"Best model F1-score: {best_row['F1']:.4f}")
print(f"Selected features used: {len(best_feature_set)}")


## 3.9 Save selected features and benchmark outputs

In [ ]:
outputs_tables_dir = Path("outputs") / "tables"
outputs_tables_dir.mkdir(parents=True, exist_ok=True)

selected_feature_outputs = {
    "selected_features_lr.csv": selected_features_lr,
    "selected_features_rf.csv": selected_features_rf,
    "selected_features_catboost.csv": selected_features_catboost,
    "selected_features_mlp.csv": selected_features_mlp
}

for filename, features in selected_feature_outputs.items():
    pd.DataFrame({"Feature": features}).to_csv(
        outputs_tables_dir / filename,
        index=False
    )

benchmark_results_df.to_csv(
    outputs_tables_dir / "model_benchmark_results.csv",
    index=False
)

print(f"Saved feature-selection and benchmark tables to: {outputs_tables_dir.resolve()}")


# 4. Final CatBoost Tuning

The primary CatBoost model is tuned using only the BorutaSHAP-selected training features. Hyperparameters are selected by stratified five-fold training F1; the hold-out test set is used only for final evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool, EFstrType
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

RANDOM_STATE = 42
outputs_tables_dir = Path("outputs") / "tables"
outputs_figures_dir = Path("outputs") / "figures"
standard_shap_dir = outputs_figures_dir / "standard_shap"
conditional_shap_dir = outputs_figures_dir / "conditional_shap"

for output_directory in (
    outputs_tables_dir,
    standard_shap_dir,
    conditional_shap_dir
):
    output_directory.mkdir(parents=True, exist_ok=True)

final_class_ratio = float(
    y_train.value_counts()[0] / y_train.value_counts()[1]
)
final_catboost_model = CatBoostClassifier(
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=[1, final_class_ratio],
    random_seed=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False,
    thread_count=1
)

# Compact CatBoost-native grid: 24 combinations evaluated with stratified
# five-fold CV on training data only. Native grid search avoids the estimator
# cloning incompatibility between CatBoost 1.2.x and scikit-learn 1.8.
final_catboost_param_grid = {
    "depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "iterations": [300, 500],
    "l2_leaf_reg": [3, 5]
}
final_catboost_train_pool = Pool(
    X_train_cb_sel,
    y_train,
    cat_features=cat_features_cb_sel
)
final_catboost_search_results = final_catboost_model.grid_search(
    param_grid=final_catboost_param_grid,
    X=final_catboost_train_pool,
    cv=5,
    partition_random_seed=RANDOM_STATE,
    calc_cv_statistics=True,
    search_by_train_test_split=False,
    refit=True,
    shuffle=True,
    stratified=True,
    verbose=False,
    plot=False
)

final_catboost_best_params = final_catboost_search_results["params"]
final_catboost_predictions = np.asarray(
    final_catboost_model.predict(X_test_cb_sel)
).ravel().astype(int)

final_catboost_confusion_matrix = confusion_matrix(
    y_test,
    final_catboost_predictions
)
final_catboost_metrics = {
    "Precision": precision_score(
        y_test, final_catboost_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, final_catboost_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, final_catboost_predictions, zero_division=0
    )
}

pd.DataFrame([final_catboost_metrics]).to_csv(
    outputs_tables_dir / "final_catboost_metrics.csv",
    index=False
)
pd.DataFrame(
    list(final_catboost_best_params.items()),
    columns=["Parameter", "Value"]
).to_csv(
    outputs_tables_dir / "final_catboost_best_params.csv",
    index=False
)
pd.DataFrame(
    final_catboost_confusion_matrix,
    index=["Actual_0", "Actual_1"],
    columns=["Predicted_0", "Predicted_1"]
).to_csv(
    outputs_tables_dir / "final_catboost_confusion_matrix.csv"
)

print("Final CatBoost best parameters:")
print(final_catboost_best_params)
print("Final CatBoost metrics:")
print(final_catboost_metrics)
print("Final CatBoost confusion matrix:")
print(final_catboost_confusion_matrix)


# 5. Standard SHAP Interpretation

Standard SHAP is the conventional baseline explanation. CatBoost's native TreeSHAP implementation is used with the final model and selected CatBoost features. SHAP describes model behavior; it is not a causal analysis.

In [ ]:
def safe_feature_filename(feature_name):
    """Create a filesystem-safe name while keeping the clinical label in plots."""
    return "".join(
        character if character.isalnum() else "_"
        for character in str(feature_name)
    ).strip("_")


def shap_display_frame(X, categorical_features):
    """Convert categories to display codes without changing model inputs."""
    display = X.copy()
    category_labels = {}
    for feature in categorical_features:
        categories = pd.Categorical(display[feature].astype(str))
        display[feature] = categories.codes.astype(float)
        category_labels[feature] = list(categories.categories)
    return display.astype(float), category_labels


def plot_shap_bar(ranking_df, value_column, output_path, title):
    ordered = ranking_df.sort_values(value_column, ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(4, 0.55 * len(ordered))))
    ax.barh(ordered["Feature"], ordered[value_column], color="#2878B5")
    ax.set_xlabel("Mean absolute SHAP value")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_shap_beeswarm(shap_values, display_X, ranking_df, output_path, title):
    ranked_features = ranking_df["Feature"].tolist()
    fig, ax = plt.subplots(figsize=(9, max(4, 0.65 * len(ranked_features))))
    rng = np.random.default_rng(RANDOM_STATE)

    for position, feature in enumerate(reversed(ranked_features)):
        feature_index = display_X.columns.get_loc(feature)
        feature_values = display_X[feature].to_numpy(dtype=float)
        value_range = np.ptp(feature_values)
        normalized_values = (
            (feature_values - np.min(feature_values)) / value_range
            if value_range > 0
            else np.full(len(feature_values), 0.5)
        )
        jitter = rng.normal(0, 0.06, len(feature_values))
        ax.scatter(
            shap_values[:, feature_index],
            position + jitter,
            c=normalized_values,
            cmap="coolwarm",
            vmin=0,
            vmax=1,
            s=32,
            alpha=0.8,
            edgecolors="none"
        )

    ax.axvline(0, color="grey", linewidth=0.8)
    ax.set_yticks(range(len(ranked_features)))
    ax.set_yticklabels(list(reversed(ranked_features)))
    ax.set_xlabel("SHAP value")
    ax.set_title(title)
    colorbar = fig.colorbar(
        plt.cm.ScalarMappable(cmap="coolwarm"),
        ax=ax,
        pad=0.02
    )
    colorbar.set_label("Relative feature value")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_shap_dependence(
    feature,
    shap_values,
    display_X,
    category_labels,
    output_path,
    title_prefix
):
    feature_index = display_X.columns.get_loc(feature)
    x_values = display_X[feature].to_numpy(dtype=float)
    y_values = shap_values[:, feature_index]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(x_values, y_values, alpha=0.8, color="#D95319")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set_xlabel(feature)
    ax.set_ylabel("SHAP value")
    ax.set_title(f"{title_prefix}: {feature}")

    if feature in category_labels:
        labels = category_labels[feature]
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=30, ha="right")

    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


standard_shap_pool = Pool(
    X_test_cb_sel,
    cat_features=cat_features_cb_sel
)
standard_shap_with_baseline = final_catboost_model.get_feature_importance(
    standard_shap_pool,
    type=EFstrType.ShapValues
)
standard_shap_values = np.asarray(standard_shap_with_baseline)[:, :-1]

standard_mean_abs_shap = np.abs(standard_shap_values).mean(axis=0)
standard_shap_ranking_df = (
    pd.DataFrame({
        "Feature": X_test_cb_sel.columns,
        "MeanAbsSHAP": standard_mean_abs_shap
    })
    .sort_values("MeanAbsSHAP", ascending=False)
    .reset_index(drop=True)
)
standard_shap_ranking_df["Standard_SHAP_Rank"] = (
    np.arange(1, len(standard_shap_ranking_df) + 1)
)
standard_shap_ranking_df.to_csv(
    outputs_tables_dir / "standard_shap_ranking.csv",
    index=False
)

standard_shap_display_X, standard_category_labels = shap_display_frame(
    X_test_cb_sel,
    cat_features_cb_sel
)

plot_shap_bar(
    standard_shap_ranking_df,
    "MeanAbsSHAP",
    standard_shap_dir / "standard_shap_bar.png",
    "Standard SHAP Feature Importance"
)
plot_shap_beeswarm(
    standard_shap_values,
    standard_shap_display_X,
    standard_shap_ranking_df,
    standard_shap_dir / "standard_shap_beeswarm.png",
    "Standard SHAP Beeswarm"
)

standard_top_features = standard_shap_ranking_df.head(
    min(5, len(standard_shap_ranking_df))
)["Feature"].tolist()

for feature in standard_top_features:
    plot_shap_dependence(
        feature,
        standard_shap_values,
        standard_shap_display_X,
        standard_category_labels,
        standard_shap_dir
        / f"standard_shap_dependence_{safe_feature_filename(feature)}.png",
        "Standard SHAP dependence"
    )

print("Top Standard SHAP features:")
print(standard_shap_ranking_df.head())


# 6. Conditional SHAP Interpretation

True model-native Conditional SHAP is not directly available for this mixed numeric/categorical CatBoost configuration. The implementation below uses an empirical nearest-neighbor conditional-expectation approximation: for each feature coalition, it finds clinically similar training records and keeps the unobserved features jointly distributed as observed in those neighbours. Exact Shapley weights are then applied across every coalition.

This is a correlation-aware explanation approximation. It is not causal proof.

In [ ]:
from itertools import combinations
from math import factorial

conditional_reference_X = X_train_cb_sel.reset_index(drop=True).copy()
conditional_numeric_features = [
    feature for feature in selected_features_catboost
    if feature not in cat_features_cb_sel
]
conditional_numeric_scales = {
    feature: max(float(conditional_reference_X[feature].std()), 1e-8)
    for feature in conditional_numeric_features
}
CONDITIONAL_NEIGHBORS = min(15, len(conditional_reference_X))


def predict_relapse_probability(X):
    return np.asarray(final_catboost_model.predict_proba(X))[:, 1]


conditional_baseline_probability = float(
    predict_relapse_probability(conditional_reference_X).mean()
)


def empirical_conditional_value(instance, coalition):
    """Approximate E[f(X) | X_coalition = instance_coalition] with KNN."""
    coalition = tuple(coalition)
    if not coalition:
        return conditional_baseline_probability

    if len(coalition) == len(selected_features_catboost):
        instance_frame = pd.DataFrame([instance], columns=selected_features_catboost)
        return float(predict_relapse_probability(instance_frame)[0])

    distances = np.zeros(len(conditional_reference_X), dtype=float)
    for feature in coalition:
        if feature in cat_features_cb_sel:
            mismatch = (
                conditional_reference_X[feature].astype(str).to_numpy()
                != str(instance[feature])
            )
            distances += mismatch.astype(float)
        else:
            difference = (
                conditional_reference_X[feature].to_numpy(dtype=float)
                - float(instance[feature])
            ) / conditional_numeric_scales[feature]
            distances += difference ** 2

    neighbour_indices = np.argsort(distances)[:CONDITIONAL_NEIGHBORS]
    conditional_samples = conditional_reference_X.iloc[
        neighbour_indices
    ].copy()

    # Fix observed coalition values while retaining the remaining features as
    # jointly observed in neighbouring training records.
    for feature in coalition:
        conditional_samples[feature] = instance[feature]

    return float(predict_relapse_probability(conditional_samples).mean())


def compute_empirical_conditional_shap(X_explain):
    features = list(selected_features_catboost)
    feature_count = len(features)
    result = np.zeros((len(X_explain), feature_count), dtype=float)

    for row_position, (_, instance) in enumerate(X_explain.iterrows()):
        coalition_values = {}

        for coalition_size in range(feature_count + 1):
            for coalition in combinations(features, coalition_size):
                coalition_values[frozenset(coalition)] = (
                    empirical_conditional_value(instance, coalition)
                )

        for feature_position, feature in enumerate(features):
            remaining_features = [
                candidate for candidate in features
                if candidate != feature
            ]
            conditional_shap_value = 0.0

            for coalition_size in range(feature_count):
                for coalition in combinations(
                    remaining_features,
                    coalition_size
                ):
                    coalition_set = frozenset(coalition)
                    coalition_with_feature = coalition_set | {feature}
                    weight = (
                        factorial(coalition_size)
                        * factorial(feature_count - coalition_size - 1)
                        / factorial(feature_count)
                    )
                    conditional_shap_value += weight * (
                        coalition_values[coalition_with_feature]
                        - coalition_values[coalition_set]
                    )

            result[row_position, feature_position] = conditional_shap_value

    return result


conditional_shap_values = compute_empirical_conditional_shap(X_test_cb_sel)
conditional_mean_abs_shap = np.abs(conditional_shap_values).mean(axis=0)
conditional_shap_ranking_df = (
    pd.DataFrame({
        "Feature": selected_features_catboost,
        "MeanAbsConditionalSHAP": conditional_mean_abs_shap
    })
    .sort_values("MeanAbsConditionalSHAP", ascending=False)
    .reset_index(drop=True)
)
conditional_shap_ranking_df["Conditional_SHAP_Rank"] = (
    np.arange(1, len(conditional_shap_ranking_df) + 1)
)
conditional_shap_ranking_df.to_csv(
    outputs_tables_dir / "conditional_shap_ranking.csv",
    index=False
)

plot_shap_bar(
    conditional_shap_ranking_df,
    "MeanAbsConditionalSHAP",
    conditional_shap_dir / "conditional_shap_bar.png",
    "Conditional SHAP Feature Importance (KNN Approximation)"
)

for feature in standard_top_features:
    if feature in selected_features_catboost:
        plot_shap_dependence(
            feature,
            conditional_shap_values,
            standard_shap_display_X,
            standard_category_labels,
            conditional_shap_dir
            / f"conditional_shap_dependence_{safe_feature_filename(feature)}.png",
            "Conditional SHAP dependence"
        )

print("Top Conditional SHAP features:")
print(conditional_shap_ranking_df.head())


# 7. Standard vs Conditional SHAP Comparison

Rank changes show whether feature importance remains stable after accounting for observed feature dependence. Stability does not imply causality.

In [ ]:
shap_comparison_df = standard_shap_ranking_df.merge(
    conditional_shap_ranking_df,
    on="Feature",
    how="inner"
)
shap_comparison_df["Rank_Change"] = (
    shap_comparison_df["Conditional_SHAP_Rank"]
    - shap_comparison_df["Standard_SHAP_Rank"]
)


def interpret_rank_change(rank_change):
    absolute_change = abs(int(rank_change))
    if absolute_change <= 1:
        return "Stable"
    if absolute_change <= 3:
        return "Moderately changed"
    return "Correlation-sensitive / changed"


shap_comparison_df["Interpretation"] = (
    shap_comparison_df["Rank_Change"].apply(interpret_rank_change)
)
shap_comparison_df = shap_comparison_df[[
    "Feature",
    "MeanAbsSHAP",
    "Standard_SHAP_Rank",
    "MeanAbsConditionalSHAP",
    "Conditional_SHAP_Rank",
    "Rank_Change",
    "Interpretation"
]].sort_values("Standard_SHAP_Rank")

shap_comparison_df.to_csv(
    outputs_tables_dir / "shap_vs_conditional_shap_comparison.csv",
    index=False
)

comparison_plot_df = shap_comparison_df.sort_values(
    "Standard_SHAP_Rank",
    ascending=False
).reset_index(drop=True)
fig, ax = plt.subplots(
    figsize=(9, max(4, 0.7 * len(comparison_plot_df)))
)

for y_position, row in comparison_plot_df.iterrows():
    ax.plot(
        [row["Standard_SHAP_Rank"], row["Conditional_SHAP_Rank"]],
        [y_position, y_position],
        color="grey",
        linewidth=1.5
    )
    ax.scatter(
        row["Standard_SHAP_Rank"],
        y_position,
        color="#2878B5",
        s=70,
        label="Standard SHAP" if y_position == 0 else None
    )
    ax.scatter(
        row["Conditional_SHAP_Rank"],
        y_position,
        color="#D95319",
        s=70,
        label="Conditional SHAP" if y_position == 0 else None
    )

ax.set_yticks(range(len(comparison_plot_df)))
ax.set_yticklabels(comparison_plot_df["Feature"])
ax.set_xlabel("Feature rank (1 = most important)")
ax.set_title("Standard vs Conditional SHAP Rankings")
ax.invert_xaxis()
ax.legend()
fig.tight_layout()
fig.savefig(
    outputs_figures_dir / "shap_vs_conditional_shap_comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

print("Final CatBoost best parameters:")
print(final_catboost_best_params)
print("Final CatBoost metrics:")
print(final_catboost_metrics)
print("Top Standard SHAP features:")
print(standard_shap_ranking_df.head())
print("Top Conditional SHAP features:")
print(conditional_shap_ranking_df.head())
print("SHAP comparison table:")
print(shap_comparison_df.to_string(index=False))


# 8. EBM Threshold Analysis

EBM was used as an interpretable threshold-discovery model to complement CatBoost and SHAP. It was not used as the primary predictive model. EBM risk curves were used to identify clinically readable threshold patterns, which were then compared with SHAP dependence plot patterns.

Thresholds identified from EBM and SHAP dependence plots are exploratory and should be interpreted as model-derived clinical signals, not definitive clinical cut-offs.

## 8.1 Prepare the EBM dataset

The EBM uses the same original feature groups selected for CatBoost, expanded to every matching one-hot column in the unscaled encoded branch. Leakage features remain excluded. Numeric thresholds transformed earlier in preprocessing are converted back to clinical scale when reported where inverse parameters are available.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.special import inv_boxcox
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

try:
    from interpret.glassbox import ExplainableBoostingClassifier
    EBM_AVAILABLE = True
except ImportError:
    EBM_AVAILABLE = False
    print(
        "The 'interpret' package is required for Section 8. "
        "Install it with: pip install interpret"
    )

outputs_tables_dir = Path("outputs") / "tables"
ebm_figures_dir = Path("outputs") / "figures" / "ebm"
outputs_tables_dir.mkdir(parents=True, exist_ok=True)
ebm_figures_dir.mkdir(parents=True, exist_ok=True)

if EBM_AVAILABLE:
    leakage_features = {"REMISSION INTERVAL", "AGE RELAPSE"}
    ebm_source_features = [
        feature for feature in selected_features_catboost
        if feature not in leakage_features
    ]

    # Each original categorical feature can map to several encoded columns.
    # Include all corresponding one-hot columns in the EBM analysis.
    ebm_features = expand_original_groups_to_encoded(
        ebm_source_features,
        X_train_tree.columns
    )

    if not ebm_features:
        raise RuntimeError("No encoded EBM features matched the CatBoost selection.")

    X_train_ebm = X_train_tree[ebm_features].copy()
    X_test_ebm = X_test_tree[ebm_features].copy()

    if leakage_features.intersection(X_train_ebm.columns):
        raise RuntimeError("Leakage feature detected in the EBM dataset.")

    print("EBM encoded features:")
    print(ebm_features)


## 8.2?8.3 Train EBM and perform sanity evaluation

Balanced training sample weights are used because `ExplainableBoostingClassifier` does not expose a direct `class_weight` argument. The test-set metrics below are a sanity check only; EBM is not replacing CatBoost as the primary predictive model.

In [ ]:
if EBM_AVAILABLE:
    ebm_sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )

    # Disable interactions so every curve and threshold belongs to one feature.
    ebm_model = ExplainableBoostingClassifier(
        interactions=0,
        max_bins=64,
        outer_bags=8,
        learning_rate=0.03,
        max_rounds=3000,
        early_stopping_rounds=100,
        min_samples_leaf=4,
        n_jobs=1,
        random_state=42
    )
    ebm_model.fit(
        X_train_ebm,
        y_train,
        sample_weight=ebm_sample_weights
    )

    ebm_predictions = np.asarray(ebm_model.predict(X_test_ebm)).astype(int)
    ebm_confusion_matrix = confusion_matrix(y_test, ebm_predictions)
    ebm_metrics = {
        "Precision": precision_score(y_test, ebm_predictions, zero_division=0),
        "Recall": recall_score(y_test, ebm_predictions, zero_division=0),
        "F1": f1_score(y_test, ebm_predictions, zero_division=0)
    }

    pd.DataFrame([ebm_metrics]).to_csv(
        outputs_tables_dir / "ebm_metrics.csv",
        index=False
    )
    pd.DataFrame(
        ebm_confusion_matrix,
        index=["Actual_0", "Actual_1"],
        columns=["Predicted_0", "Predicted_1"]
    ).to_csv(outputs_tables_dir / "ebm_confusion_matrix.csv")


## 8.4?8.5 Global explanation and EBM curves

Features are prioritized using Standard SHAP rank and stability in the Standard-vs-Conditional SHAP comparison. Every matching encoded EBM term is retained.

In [ ]:
if EBM_AVAILABLE:
    ebm_global_explanation = ebm_model.explain_global(
        name="SLE Relapse EBM Global Explanation"
    )
    ebm_term_importances = np.asarray(ebm_model.term_importances())

    ebm_feature_importance_df = (
        pd.DataFrame({
            "Feature": ebm_model.term_names_,
            "Source_Feature": [
                encoded_to_original_feature(feature)
                for feature in ebm_model.term_names_
            ],
            "Importance": ebm_term_importances
        })
        .sort_values("Importance", ascending=False)
        .reset_index(drop=True)
    )
    ebm_feature_importance_df["EBM_Importance_Rank"] = (
        np.arange(1, len(ebm_feature_importance_df) + 1)
    )
    ebm_feature_importance_df.to_csv(
        outputs_tables_dir / "ebm_feature_importance.csv",
        index=False
    )

    importance_plot_df = ebm_feature_importance_df.sort_values(
        "Importance",
        ascending=True
    )
    fig, ax = plt.subplots(
        figsize=(9, max(4, 0.55 * len(importance_plot_df)))
    )
    ax.barh(
        importance_plot_df["Feature"],
        importance_plot_df["Importance"],
        color="#2878B5"
    )
    ax.set_xlabel("Mean absolute EBM contribution")
    ax.set_title("EBM Global Feature Importance")
    fig.tight_layout()
    fig.savefig(
        ebm_figures_dir / "ebm_global_feature_importance.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)

    stable_shap_sources = set(
        shap_comparison_df.loc[
            shap_comparison_df["Interpretation"] == "Stable",
            "Feature"
        ]
    )
    ranked_shap_sources = standard_shap_ranking_df.head(8)[
        "Feature"
    ].tolist()
    selected_ebm_sources = []
    for feature in ranked_shap_sources + list(stable_shap_sources):
        if feature in ebm_source_features and feature not in selected_ebm_sources:
            selected_ebm_sources.append(feature)

    selected_ebm_curve_features = [
        feature for feature in ebm_model.term_names_
        if encoded_to_original_feature(feature) in selected_ebm_sources
    ]

    def ebm_curve_data(feature):
        term_index = ebm_model.term_names_.index(feature)
        term_data = ebm_global_explanation.data(term_index)
        boundaries = np.asarray(term_data["names"], dtype=float)
        scores = np.asarray(term_data["scores"], dtype=float)

        if len(boundaries) == len(scores) + 1:
            x_values = (boundaries[:-1] + boundaries[1:]) / 2
        else:
            x_values = boundaries[:len(scores)]

        clinical_boundaries = boundaries.copy()
        clinical_x_values = x_values.copy()
        axis_label = feature

        # LOWEST HB was Box?Cox transformed in the shared preprocessing branch.
        # Convert its EBM axis and reported thresholds back to clinical units.
        if feature in lambdas:
            transform_state = lambdas[feature]
            clinical_boundaries = (
                inv_boxcox(boundaries, transform_state["lambda"])
                - transform_state["shift"]
            )
            clinical_x_values = (
                inv_boxcox(x_values, transform_state["lambda"])
                - transform_state["shift"]
            )
            axis_label = f"{feature} (clinical scale)"

        return {
            "term_index": term_index,
            "boundaries": clinical_boundaries,
            "x_values": clinical_x_values,
            "scores": scores,
            "axis_label": axis_label
        }

    ebm_curve_cache = {}
    for feature in selected_ebm_curve_features:
        curve = ebm_curve_data(feature)
        ebm_curve_cache[feature] = curve

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.plot(
            curve["x_values"],
            curve["scores"],
            marker="o",
            color="#D95319"
        )
        ax.axhline(0, color="grey", linewidth=0.8)
        ax.set_xlabel(curve["axis_label"])
        ax.set_ylabel("EBM contribution to relapse log-odds")
        ax.set_title(f"EBM Risk Curve: {feature}")
        fig.tight_layout()
        fig.savefig(
            ebm_figures_dir
            / f"ebm_curve_{safe_feature_filename(feature)}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(fig)


## 8.6 Extract exploratory EBM thresholds

Thresholds are identified first from a transition across zero EBM contribution. If no crossing exists, the largest adjacent score change is used. Confidence reflects the clarity and magnitude of this model-derived transition.

In [ ]:
if EBM_AVAILABLE:
    def derive_ebm_threshold(feature, curve):
        boundaries = curve["boundaries"]
        scores = curve["scores"]

        if len(scores) < 2 or np.allclose(scores, scores[0]):
            return {
                "Feature": feature,
                "EBM_Threshold": np.nan,
                "Threshold_Direction": "Non-linear pattern; no clear single threshold",
                "EBM_Interpretation": (
                    "The EBM curve does not show a distinct risk transition."
                ),
                "Confidence_Level": "Weak / exploratory"
            }

        adjacent_changes = np.abs(np.diff(scores))
        crossing_indices = np.where(
            (scores[:-1] <= 0) & (scores[1:] > 0)
            | (scores[:-1] >= 0) & (scores[1:] < 0)
        )[0]

        if len(crossing_indices) > 0:
            transition_index = crossing_indices[
                np.argmax(adjacent_changes[crossing_indices])
            ]
            used_zero_crossing = True
        else:
            transition_index = int(np.argmax(adjacent_changes))
            used_zero_crossing = False

        threshold = float(boundaries[transition_index + 1])
        lower_score = float(np.mean(scores[:transition_index + 1]))
        upper_score = float(np.mean(scores[transition_index + 1:]))
        change_magnitude = float(adjacent_changes[transition_index])

        if upper_score > lower_score:
            direction = "Above threshold associated with higher predicted relapse risk"
        elif lower_score > upper_score:
            direction = "Below threshold associated with higher predicted relapse risk"
        else:
            direction = "Non-linear pattern; no clear single threshold"

        if used_zero_crossing and change_magnitude >= 0.25:
            confidence = "Clear"
        elif change_magnitude >= 0.10:
            confidence = "Moderate"
        else:
            confidence = "Weak / exploratory"

        interpretation = (
            f"The largest interpretable EBM risk transition occurs near "
            f"{threshold:.3g}; {direction.lower()}."
        )
        return {
            "Feature": feature,
            "EBM_Threshold": threshold,
            "Threshold_Direction": direction,
            "EBM_Interpretation": interpretation,
            "Confidence_Level": confidence
        }

    ebm_threshold_summary_df = pd.DataFrame([
        derive_ebm_threshold(feature, ebm_curve_cache[feature])
        for feature in selected_ebm_curve_features
    ])
    ebm_threshold_summary_df.to_csv(
        outputs_tables_dir / "ebm_threshold_summary.csv",
        index=False
    )


## 8.7 EBM vs SHAP threshold comparison

SHAP dependence thresholds are extracted programmatically from the existing test-set Standard SHAP values. For continuous features, the method prioritizes a smoothed SHAP zero-crossing and otherwise uses the largest adjacent SHAP change. For categorical features, each one-hot EBM term is compared with the mean SHAP effect of the corresponding original category. Numeric agreement uses both threshold proximity and risk direction; categorical agreement uses category-effect direction. These comparisons remain exploratory because the hold-out set is small.

In [ ]:
if EBM_AVAILABLE:
    def to_clinical_scale(feature, values):
        """Invert training-fitted numeric transformations when available."""
        values = np.asarray(values, dtype=float)
        if feature in lambdas:
            transform_state = lambdas[feature]
            return (
                inv_boxcox(values, transform_state["lambda"])
                - transform_state["shift"]
            )
        return values


    def derive_shap_dependence_threshold(feature):
        """Use the midpoint of the observed raw SHAP sign-change interval."""
        feature_index = X_test_cb_sel.columns.get_loc(feature)
        clinical_values = to_clinical_scale(
            feature,
            X_test_cb_sel[feature].to_numpy(dtype=float)
        )
        shap_values_for_feature = standard_shap_values[:, feature_index]

        dependence = (
            pd.DataFrame({
                "Feature_Value": clinical_values,
                "SHAP_Value": shap_values_for_feature
            })
            .groupby("Feature_Value", as_index=False)["SHAP_Value"]
            .mean()
            .sort_values("Feature_Value")
            .reset_index(drop=True)
        )

        positive_rows = dependence.loc[dependence["SHAP_Value"] > 0]
        negative_rows = dependence.loc[dependence["SHAP_Value"] < 0]

        transition_lower = np.nan
        transition_upper = np.nan
        lower_effect = np.nan
        upper_effect = np.nan

        # Primary clinical interpretation: last observed positive-SHAP value,
        # followed by the first higher observed negative-SHAP value.
        if not positive_rows.empty:
            last_positive_value = float(positive_rows["Feature_Value"].max())
            later_negative_rows = negative_rows.loc[
                negative_rows["Feature_Value"] > last_positive_value
            ]
            if not later_negative_rows.empty:
                transition_lower = last_positive_value
                transition_upper = float(
                    later_negative_rows["Feature_Value"].min()
                )
                lower_effect = float(
                    dependence.loc[
                        dependence["Feature_Value"] == transition_lower,
                        "SHAP_Value"
                    ].iloc[0]
                )
                upper_effect = float(
                    dependence.loc[
                        dependence["Feature_Value"] == transition_upper,
                        "SHAP_Value"
                    ].iloc[0]
                )

        # General fallback for a negative-to-positive observed transition.
        if pd.isna(transition_lower) and not negative_rows.empty:
            last_negative_value = float(negative_rows["Feature_Value"].max())
            later_positive_rows = positive_rows.loc[
                positive_rows["Feature_Value"] > last_negative_value
            ]
            if not later_positive_rows.empty:
                transition_lower = last_negative_value
                transition_upper = float(
                    later_positive_rows["Feature_Value"].min()
                )
                lower_effect = float(
                    dependence.loc[
                        dependence["Feature_Value"] == transition_lower,
                        "SHAP_Value"
                    ].iloc[0]
                )
                upper_effect = float(
                    dependence.loc[
                        dependence["Feature_Value"] == transition_upper,
                        "SHAP_Value"
                    ].iloc[0]
                )

        if pd.isna(transition_lower) or pd.isna(transition_upper):
            return {
                "Source_Feature": feature,
                "SHAP_Dependence_Threshold": np.nan,
                "SHAP_Transition_Lower": np.nan,
                "SHAP_Transition_Upper": np.nan,
                "SHAP_Threshold_Direction": (
                    "Non-linear pattern; no clear observed sign transition"
                ),
                "SHAP_Threshold_Method": "No raw observed sign change",
                "SHAP_Threshold_Confidence": "Weak / exploratory"
            }

        threshold = float((transition_lower + transition_upper) / 2)
        if lower_effect > 0 and upper_effect < 0:
            direction = (
                "Below threshold associated with higher predicted relapse risk"
            )
        elif lower_effect < 0 and upper_effect > 0:
            direction = (
                "Above threshold associated with higher predicted relapse risk"
            )
        else:
            direction = "Non-linear pattern; no clear single threshold"

        transition_width = transition_upper - transition_lower
        observed_range = max(
            float(dependence["Feature_Value"].max()
                  - dependence["Feature_Value"].min()),
            1e-8
        )
        relative_width = transition_width / observed_range
        confidence = (
            "Moderate" if relative_width <= 0.15
            else "Weak / exploratory"
        )

        # Plot raw observations and raw mean SHAP only. The shaded interval is
        # the observed transition; the dashed line is its midpoint estimate.
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(
            clinical_values,
            shap_values_for_feature,
            alpha=0.65,
            color="#D95319",
            label="Patient-level SHAP"
        )
        ax.axhline(0, color="grey", linewidth=0.8)
        ax.axvspan(
            transition_lower,
            transition_upper,
            color="grey",
            alpha=0.18,
            label=(
                f"Observed transition {transition_lower:.2f}?"
                f"{transition_upper:.2f}"
            )
        )
        ax.axvline(
            threshold,
            color="black",
            linestyle="--",
            label=f"Midpoint estimate ? {threshold:.2f}"
        )
        ax.set_xlabel(f"{feature} (clinical scale)")
        ax.set_ylabel("Standard SHAP value")
        ax.set_title(f"SHAP Observed Transition: {feature}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            standard_shap_dir
            / f"standard_shap_threshold_{safe_feature_filename(feature)}.png",
            dpi=300,
            bbox_inches="tight"
        )
        plt.close(fig)

        return {
            "Source_Feature": feature,
            "SHAP_Dependence_Threshold": threshold,
            "SHAP_Transition_Lower": transition_lower,
            "SHAP_Transition_Upper": transition_upper,
            "SHAP_Threshold_Direction": direction,
            "SHAP_Threshold_Method": "Raw observed sign-change midpoint",
            "SHAP_Threshold_Confidence": confidence
        }


    def derive_categorical_shap_effect(source_feature, encoded_feature):
        """Compare the SHAP effect of one original category against all others."""
        category_label = encoded_feature[len(source_feature) + 1:]
        source_values = X_test_cb_sel[source_feature].astype(str)
        feature_index = X_test_cb_sel.columns.get_loc(source_feature)
        source_shap = standard_shap_values[:, feature_index]
        category_mask = source_values.to_numpy() == category_label

        if category_mask.sum() == 0 or (~category_mask).sum() == 0:
            direction = "Category effect not assessable in hold-out data"
            confidence = "Weak / exploratory"
            category_effect = np.nan
        else:
            category_mean = float(np.mean(source_shap[category_mask]))
            other_mean = float(np.mean(source_shap[~category_mask]))
            category_effect = category_mean - other_mean
            direction = (
                "Above threshold associated with higher predicted relapse risk"
                if category_effect > 0
                else "Below threshold associated with higher predicted relapse risk"
            )
            confidence = (
                "Moderate" if abs(category_effect) >= 0.10
                else "Weak / exploratory"
            )

        return {
            "Feature": encoded_feature,
            "Source_Feature": source_feature,
            "SHAP_Dependence_Threshold": 0.5,
            "SHAP_Threshold_Direction": direction,
            "SHAP_Threshold_Method": (
                f"Category {category_label} mean SHAP effect vs other categories"
            ),
            "SHAP_Threshold_Confidence": confidence,
            "SHAP_Category_Effect": category_effect
        }


    numeric_shap_thresholds = {
        feature: derive_shap_dependence_threshold(feature)
        for feature in selected_ebm_sources
        if feature not in cat_features_cb_sel
    }

    shap_threshold_rows = []
    for ebm_feature in ebm_threshold_summary_df["Feature"]:
        source_feature = encoded_to_original_feature(ebm_feature)
        if source_feature in numeric_shap_thresholds:
            row = {"Feature": ebm_feature}
            row.update(numeric_shap_thresholds[source_feature])
        else:
            row = derive_categorical_shap_effect(
                source_feature,
                ebm_feature
            )
        shap_threshold_rows.append(row)

    standard_shap_threshold_df = pd.DataFrame(shap_threshold_rows)
    standard_shap_threshold_df.to_csv(
        outputs_tables_dir / "standard_shap_thresholds.csv",
        index=False
    )

    def regenerate_threshold_dependence_plot(
        feature,
        shap_matrix,
        output_path,
        title_prefix
    ):
        """Orange patient-level points with only the threshold annotation."""
        source_feature = encoded_to_original_feature(feature)
        source_index = X_test_cb_sel.columns.get_loc(source_feature)
        y_values = shap_matrix[:, source_index]

        if source_feature in cat_features_cb_sel:
            labels = standard_category_labels[source_feature]
            x_values = standard_shap_display_X[source_feature].to_numpy(dtype=float)
            threshold = 0.5
            lower = upper = np.nan
            rng = np.random.default_rng(RANDOM_STATE)
            x_plot = x_values + rng.normal(0, 0.035, len(x_values))
        else:
            x_values = to_clinical_scale(
                source_feature,
                X_test_cb_sel[source_feature].to_numpy(dtype=float)
            )
            threshold_row = standard_shap_threshold_df.loc[
                standard_shap_threshold_df["Source_Feature"] == source_feature
            ].iloc[0]
            threshold = float(threshold_row["SHAP_Dependence_Threshold"])
            lower = float(threshold_row["SHAP_Transition_Lower"])
            upper = float(threshold_row["SHAP_Transition_Upper"])
            labels = None
            x_plot = x_values

        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(
            x_plot,
            y_values,
            s=48,
            alpha=0.78,
            color="#E8753D",
            edgecolor="#C85A28",
            linewidth=0.8,
            label="Patient-level SHAP"
        )
        ax.axhline(0, color="grey", linewidth=0.8)
        if not pd.isna(lower) and not pd.isna(upper):
            ax.axvspan(
                lower,
                upper,
                color="grey",
                alpha=0.14,
                label=f"Observed transition {lower:.2f}?{upper:.2f}"
            )
        ax.axvline(
            threshold,
            color="black",
            linestyle="--",
            linewidth=1.5,
            label=f"Threshold ? {threshold:.2f}"
        )
        ax.set_xlabel(
            f"{source_feature} (clinical scale)"
            if source_feature not in cat_features_cb_sel
            else source_feature
        )
        ax.set_ylabel(f"{title_prefix} SHAP value")
        ax.set_title(f"{title_prefix} SHAP Dependence: {source_feature}")
        if labels is not None:
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels, rotation=30, ha="right")
        ax.legend()
        fig.tight_layout()
        fig.savefig(output_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

    # Overwrite every dependence plot with the same clean visual convention:
    # orange patient-level points and the relevant vertical threshold only.
    for source_feature in standard_top_features:
        regenerate_threshold_dependence_plot(
            source_feature,
            standard_shap_values,
            standard_shap_dir
            / f"standard_shap_dependence_{safe_feature_filename(source_feature)}.png",
            "Standard"
        )
        regenerate_threshold_dependence_plot(
            source_feature,
            conditional_shap_values,
            conditional_shap_dir
            / f"conditional_shap_dependence_{safe_feature_filename(source_feature)}.png",
            "Conditional"
        )

    # Keep the threshold-specific filename as the same clean Standard plot.
    regenerate_threshold_dependence_plot(
        "LOWEST HB",
        standard_shap_values,
        standard_shap_dir / "standard_shap_threshold_LOWEST_HB.png",
        "Standard"
    )

    ebm_vs_shap_threshold_df = ebm_threshold_summary_df.merge(
        standard_shap_threshold_df,
        on="Feature",
        how="left"
    )

    def compare_threshold_row(row):
        shap_threshold = row["SHAP_Dependence_Threshold"]
        ebm_threshold = row["EBM_Threshold"]
        shap_direction = row["SHAP_Threshold_Direction"]
        ebm_direction = row["Threshold_Direction"]

        if pd.isna(shap_threshold) or pd.isna(ebm_threshold):
            return pd.Series({
                "Agreement": "Not assessable",
                "Interpretation": (
                    "A numeric threshold could not be estimated for both methods."
                )
            })

        if "not assessable" in str(shap_direction).lower():
            return pd.Series({
                "Agreement": "Not assessable",
                "Interpretation": shap_direction
            })

        direction_agrees = shap_direction == ebm_direction
        source_feature = row["Source_Feature"]

        if source_feature in cat_features_cb_sel:
            if direction_agrees:
                agreement = "Agree"
                interpretation = (
                    "Both methods assign the same risk direction to this "
                    "categorical level (the 0.5 threshold denotes dummy absence/presence)."
                )
            else:
                agreement = "Disagree"
                interpretation = (
                    "EBM and SHAP assign opposite risk directions to this "
                    "categorical level."
                )
            return pd.Series({
                "Agreement": agreement,
                "Interpretation": interpretation
            })

        absolute_difference = abs(float(shap_threshold) - float(ebm_threshold))
        reference_scale = max(abs(float(ebm_threshold)), 1e-8)
        relative_difference = absolute_difference / reference_scale

        if direction_agrees and relative_difference <= 0.10:
            agreement = "Agree"
        elif direction_agrees and relative_difference <= 0.20:
            agreement = "Partially agree"
        else:
            agreement = "Disagree"

        interpretation = (
            f"SHAP threshold ? {float(shap_threshold):.2f}; "
            f"EBM threshold ? {float(ebm_threshold):.2f}. "
            f"Risk direction {'matches' if direction_agrees else 'differs'}; "
            f"relative threshold difference is {relative_difference:.1%}."
        )
        return pd.Series({
            "Agreement": agreement,
            "Interpretation": interpretation
        })

    agreement_results = ebm_vs_shap_threshold_df.apply(
        compare_threshold_row,
        axis=1
    )
    ebm_vs_shap_threshold_df[["Agreement", "Interpretation"]] = (
        agreement_results
    )
    ebm_vs_shap_threshold_df = ebm_vs_shap_threshold_df[[
        "Feature",
        "Source_Feature",
        "SHAP_Dependence_Threshold",
        "EBM_Threshold",
        "SHAP_Threshold_Direction",
        "Threshold_Direction",
        "Agreement",
        "Interpretation"
    ]]
    ebm_vs_shap_threshold_df.to_csv(
        outputs_tables_dir / "ebm_vs_shap_threshold_comparison.csv",
        index=False
    )

    print("EBM sanity metrics:")
    print(ebm_metrics)
    print("EBM confusion matrix:")
    print(ebm_confusion_matrix)
    print("Top EBM important features:")
    print(ebm_feature_importance_df.head(8).to_string(index=False))
    print("SHAP dependence thresholds:")
    print(standard_shap_threshold_df.to_string(index=False))
    print("EBM threshold summary:")
    print(ebm_threshold_summary_df.to_string(index=False))
    print("EBM vs SHAP threshold comparison:")
    print(ebm_vs_shap_threshold_df.to_string(index=False))


# 9. DML Exploratory Analysis

DML was used to explore possible causal signals after adjustment for observed covariates, not to prove causality.

DML is not used as a prediction model and is not included in the model benchmark table. It is used as an exploratory causal-inference layer to evaluate whether important predictors retain adjusted associations with relapse after controlling for other observed selected features.

DML results are exploratory and should not be interpreted as definitive causal proof due to the retrospective design, small sample size, and possible unmeasured confounding.

## 9.1 Prepare DML inputs

Candidate features are restricted to important and stable source features identified by Standard SHAP, Conditional SHAP, and EBM. Original categorical features are expanded to their encoded columns. The analysis uses the leakage-safe training set only, preserving the hold-out test set.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold

try:
    from econml.dml import LinearDML
    ECONML_AVAILABLE = True
except ImportError:
    ECONML_AVAILABLE = False
    print(
        "econml is not installed (install with: pip install econml). "
        "Using the documented cross-fitted residualization DML fallback."
    )

DML_RANDOM_STATE = 42
dml_tables_dir = Path("outputs") / "tables"
dml_figures_dir = Path("outputs") / "figures"
dml_tables_dir.mkdir(parents=True, exist_ok=True)
dml_figures_dir.mkdir(parents=True, exist_ok=True)

dml_leakage_features = {"REMISSION INTERVAL", "AGE RELAPSE"}

standard_ranked_sources = standard_shap_ranking_df["Feature"].tolist()
conditional_ranked_sources = conditional_shap_ranking_df["Feature"].tolist()
stable_sources = shap_comparison_df.loc[
    shap_comparison_df["Interpretation"] == "Stable",
    "Feature"
].tolist()
ebm_sources = [
    encoded_to_original_feature(feature)
    for feature in ebm_threshold_summary_df["Feature"].tolist()
]

# Preserve source-feature priority while removing duplicates.
dml_candidate_sources = []
for source_feature in (
    standard_ranked_sources
    + conditional_ranked_sources
    + stable_sources
    + ebm_sources
):
    if (
        source_feature not in dml_leakage_features
        and source_feature not in dml_candidate_sources
    ):
        dml_candidate_sources.append(source_feature)

# Expand selected original categorical features to all matching one-hot terms.
dml_candidate_features = expand_original_groups_to_encoded(
    dml_candidate_sources,
    X_train_tree.columns
)[:8]

if len(dml_candidate_features) < 2:
    raise RuntimeError(
        "DML requires at least two candidate features for treatment/control roles."
    )

X_dml = X_train_tree[dml_candidate_features].copy()
y_dml = y_train.copy()

# Restore original clinical units for continuous treatments where the shared
# tree branch contains a Box-Cox/log transformation. This makes a continuous
# DML coefficient interpretable per clinical unit while retaining encoded,
# unscaled categorical controls.
for feature in dml_candidate_features:
    source_feature = encoded_to_original_feature(feature)
    if (
        feature == source_feature
        and source_feature in X_train_imputed.columns
        and source_feature not in categorical_cols
    ):
        X_dml[feature] = X_train_imputed.loc[X_dml.index, source_feature]

if dml_leakage_features.intersection(X_dml.columns):
    raise RuntimeError("Leakage feature detected in DML inputs.")

print("DML candidate features:")
print(dml_candidate_features)


## 9.2 Run cross-fitted residualization DML per feature

`econml` is optional. When unavailable, the fallback follows the DML residualization principle with five-fold cross-fitting: CatBoost predicts the outcome and treatment from controls in held-out folds, then HC3-robust regression estimates the association between the two residuals. Binary treatment effects are approximate adjusted risk-difference signals; continuous effects are per clinical unit.

In [ ]:
def dml_feature_type(series):
    unique_values = set(pd.Series(series).dropna().unique().tolist())
    return "Binary" if unique_values.issubset({0, 1}) else "Continuous"


def make_binary_catboost(y_values, seed):
    counts = pd.Series(y_values).value_counts()
    if len(counts) < 2:
        return None
    negative_count = float(counts.get(0, 0))
    positive_count = float(counts.get(1, 0))
    positive_weight = (
        negative_count / positive_count if positive_count > 0 else 1.0
    )
    return CatBoostClassifier(
        iterations=300,
        depth=3,
        learning_rate=0.03,
        loss_function="Logloss",
        class_weights=[1, positive_weight],
        random_seed=seed,
        verbose=0,
        allow_writing_files=False,
        thread_count=1
    )


def run_residualization_dml(treatment_feature):
    treatment = X_dml[treatment_feature].astype(float).to_numpy()
    controls = X_dml.drop(columns=[treatment_feature])
    outcome = y_dml.astype(float).to_numpy()
    feature_type = dml_feature_type(treatment)

    outcome_predictions = np.zeros(len(X_dml), dtype=float)
    treatment_predictions = np.zeros(len(X_dml), dtype=float)

    cross_fitter = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=DML_RANDOM_STATE
    )

    for fold_number, (train_indices, validation_indices) in enumerate(
        cross_fitter.split(controls, y_dml),
        start=1
    ):
        controls_train = controls.iloc[train_indices]
        controls_validation = controls.iloc[validation_indices]
        outcome_train = y_dml.iloc[train_indices]
        treatment_train = treatment[train_indices]

        outcome_model = make_binary_catboost(
            outcome_train,
            DML_RANDOM_STATE + fold_number
        )
        outcome_model.fit(controls_train, outcome_train)
        outcome_predictions[validation_indices] = outcome_model.predict_proba(
            controls_validation
        )[:, 1]

        if feature_type == "Binary":
            treatment_model = make_binary_catboost(
                treatment_train,
                DML_RANDOM_STATE + 100 + fold_number
            )
            if treatment_model is None:
                treatment_predictions[validation_indices] = np.mean(
                    treatment_train
                )
            else:
                treatment_model.fit(controls_train, treatment_train.astype(int))
                treatment_predictions[validation_indices] = (
                    treatment_model.predict_proba(controls_validation)[:, 1]
                )
        else:
            treatment_model = CatBoostRegressor(
                iterations=300,
                depth=3,
                learning_rate=0.03,
                loss_function="RMSE",
                random_seed=DML_RANDOM_STATE + 100 + fold_number,
                verbose=0,
                allow_writing_files=False,
                thread_count=1
            )
            treatment_model.fit(controls_train, treatment_train)
            treatment_predictions[validation_indices] = treatment_model.predict(
                controls_validation
            )

    outcome_residual = outcome - outcome_predictions
    treatment_residual = treatment - treatment_predictions

    if np.std(treatment_residual) < 1e-8:
        return {
            "Feature": treatment_feature,
            "Feature_Type": feature_type,
            "Estimated_Effect": np.nan,
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "P_Value": np.nan,
            "Direction": "No clear adjusted signal",
            "Interpretation": (
                "Treatment residual had insufficient variation for estimation."
            )
        }

    # No intercept: cross-fitted nuisance residuals are centered by construction.
    final_stage = sm.OLS(
        outcome_residual,
        treatment_residual.reshape(-1, 1)
    ).fit(cov_type="HC3")

    estimated_effect = float(final_stage.params[0])
    ci_lower, ci_upper = [
        float(value) for value in final_stage.conf_int(alpha=0.05)[0]
    ]
    p_value = float(final_stage.pvalues[0])

    if p_value < 0.05 and ci_lower > 0:
        direction = "Positive adjusted signal with relapse"
        interpretation = "Positive adjusted signal with relapse"
    elif p_value < 0.05 and ci_upper < 0:
        direction = "Negative/protective adjusted signal with relapse"
        interpretation = "Negative adjusted signal with relapse"
    else:
        direction = "No clear adjusted signal"
        interpretation = "No clear adjusted signal"

    return {
        "Feature": treatment_feature,
        "Feature_Type": feature_type,
        "Estimated_Effect": estimated_effect,
        "CI_Lower": ci_lower,
        "CI_Upper": ci_upper,
        "P_Value": p_value,
        "Direction": direction,
        "Interpretation": interpretation
    }


dml_results_df = pd.DataFrame([
    run_residualization_dml(feature)
    for feature in dml_candidate_features
])


## 9.3?9.4 Save and summarize DML results

Effects are exploratory adjusted signals. Statistical uncertainty remains substantial because the sample is small and candidate features were selected post hoc.

In [ ]:
dml_results_df.to_csv(
    dml_tables_dir / "dml_exploratory_results.csv",
    index=False
)

plot_results = dml_results_df.sort_values(
    "Estimated_Effect",
    ascending=True,
    na_position="first"
).reset_index(drop=True)

fig, ax = plt.subplots(
    figsize=(9, max(4, 0.65 * len(plot_results)))
)
valid_results = plot_results["Estimated_Effect"].notna()
y_positions = np.arange(len(plot_results))

ax.scatter(
    plot_results.loc[valid_results, "Estimated_Effect"],
    y_positions[valid_results],
    color="#2878B5",
    s=65,
    zorder=3
)
for position, row in plot_results.loc[valid_results].iterrows():
    ax.plot(
        [row["CI_Lower"], row["CI_Upper"]],
        [position, position],
        color="#2878B5",
        linewidth=2
    )

ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels(plot_results["Feature"])
ax.set_xlabel("Exploratory adjusted effect (95% HC3 CI)")
ax.set_title("DML Exploratory Effect Summary")
fig.tight_layout()
fig.savefig(
    dml_figures_dir / "dml_effect_summary.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)

positive_adjusted_signals = dml_results_df.loc[
    dml_results_df["Direction"] == "Positive adjusted signal with relapse",
    "Feature"
].tolist()
negative_adjusted_signals = dml_results_df.loc[
    dml_results_df["Direction"]
    == "Negative/protective adjusted signal with relapse",
    "Feature"
].tolist()
no_clear_adjusted_signals = dml_results_df.loc[
    dml_results_df["Direction"] == "No clear adjusted signal",
    "Feature"
].tolist()

print("DML candidate features:")
print(dml_candidate_features)
print("DML results table:")
print(dml_results_df.to_string(index=False))
print("Positive adjusted signals:")
print(positive_adjusted_signals)
print("Negative adjusted signals:")
print(negative_adjusted_signals)
print("Features with no clear signal:")
print(no_clear_adjusted_signals)


# 10. REMISSION INTERVAL Sensitivity Analysis

REMISSION INTERVAL was excluded from the primary predictive model due to possible outcome-timing leakage. It was included only in this secondary sensitivity analysis to examine clinically meaningful remission-duration threshold patterns.

SHAP and EBM threshold findings for REMISSION INTERVAL are exploratory and should not be interpreted as causal effects or definitive clinical cut-offs.

## 10.1 Create the separate sensitivity datasets

Only the final CatBoost-selected features and `REMISSION INTERVAL` are included. The original primary model, benchmark, feature selection, and DML datasets remain unchanged. The existing train/test indices are reused, and any remission-value imputation is fitted on training data only.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool, EFstrType
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight

remission_sensitivity_tables_dir = Path("outputs") / "tables"
remission_sensitivity_figures_dir = (
    Path("outputs") / "figures" / "remission_interval_sensitivity"
)
remission_sensitivity_tables_dir.mkdir(parents=True, exist_ok=True)
remission_sensitivity_figures_dir.mkdir(parents=True, exist_ok=True)

# Locate the original workbook without modifying the primary `data` object.
remission_source_candidates = [
    Path("ML_DATA SLE RELAPSE SSM.xlsx"),
    Path("Dataset") / "ML_DATA SLE RELAPSE SSM.xlsx",
    Path("..") / "Dataset" / "ML_DATA SLE RELAPSE SSM.xlsx"
]
remission_source_path = next(
    (candidate for candidate in remission_source_candidates if candidate.exists()),
    None
)
if remission_source_path is None:
    raise FileNotFoundError(
        "Could not locate ML_DATA SLE RELAPSE SSM.xlsx for sensitivity analysis."
    )

remission_source = pd.read_excel(
    remission_source_path,
    usecols=["REMISSION INTERVAL"]
)["REMISSION INTERVAL"]

# Reuse the primary split indices exactly. Fit imputation on training only.
remission_train_raw = remission_source.loc[X_train_raw.index].astype(float)
remission_test_raw = remission_source.loc[X_test_raw.index].astype(float)
remission_train_median = float(remission_train_raw.median())
remission_train_values = remission_train_raw.fillna(remission_train_median)
remission_test_values = remission_test_raw.fillna(remission_train_median)

remission_sensitivity_features = list(selected_features_catboost) + [
    "REMISSION INTERVAL"
]
if "AGE RELAPSE" in remission_sensitivity_features:
    raise RuntimeError("AGE RELAPSE must not enter the sensitivity branch.")

X_train_remission_sens_cb = X_train_cb_sel.copy()
X_test_remission_sens_cb = X_test_cb_sel.copy()
X_train_remission_sens_cb["REMISSION INTERVAL"] = remission_train_values
X_test_remission_sens_cb["REMISSION INTERVAL"] = remission_test_values

remission_selected_encoded_features = expand_original_groups_to_encoded(
    selected_features_catboost,
    X_train_tree.columns
)
X_train_remission_sens_encoded = X_train_tree[
    remission_selected_encoded_features
].copy()
X_test_remission_sens_encoded = X_test_tree[
    remission_selected_encoded_features
].copy()
X_train_remission_sens_encoded["REMISSION INTERVAL"] = remission_train_values
X_test_remission_sens_encoded["REMISSION INTERVAL"] = remission_test_values

remission_sensitivity_cat_features = [
    feature for feature in selected_features_catboost
    if feature in cat_features_cb
]

print("REMISSION INTERVAL sensitivity features:")
print(remission_sensitivity_features)


## 10.2 Train the separate CatBoost sensitivity model

This is not the primary predictive model. It is evaluated only to confirm that the explanatory sensitivity branch is operational.

In [ ]:
remission_class_ratio = float(
    y_train.value_counts()[0] / y_train.value_counts()[1]
)
remission_model_params = {
    "depth": int(final_catboost_best_params.get("depth", 3)),
    "learning_rate": float(
        final_catboost_best_params.get("learning_rate", 0.03)
    ),
    "iterations": int(final_catboost_best_params.get("iterations", 300)),
    "l2_leaf_reg": float(final_catboost_best_params.get("l2_leaf_reg", 3))
}

remission_sensitivity_catboost_model = CatBoostClassifier(
    **remission_model_params,
    loss_function="Logloss",
    eval_metric="F1",
    class_weights=[1, remission_class_ratio],
    random_seed=42,
    verbose=0,
    allow_writing_files=False,
    thread_count=1
)
remission_sensitivity_catboost_model.fit(
    X_train_remission_sens_cb,
    y_train,
    cat_features=remission_sensitivity_cat_features
)

remission_sensitivity_predictions = np.asarray(
    remission_sensitivity_catboost_model.predict(
        X_test_remission_sens_cb
    )
).ravel().astype(int)
remission_sensitivity_confusion_matrix = confusion_matrix(
    y_test,
    remission_sensitivity_predictions
)
remission_sensitivity_metrics = {
    "Precision": precision_score(
        y_test, remission_sensitivity_predictions, zero_division=0
    ),
    "Recall": recall_score(
        y_test, remission_sensitivity_predictions, zero_division=0
    ),
    "F1": f1_score(
        y_test, remission_sensitivity_predictions, zero_division=0
    )
}

pd.DataFrame([remission_sensitivity_metrics]).to_csv(
    remission_sensitivity_tables_dir
    / "remission_interval_sensitivity_metrics.csv",
    index=False
)
pd.DataFrame(
    remission_sensitivity_confusion_matrix,
    index=["Actual_0", "Actual_1"],
    columns=["Predicted_0", "Predicted_1"]
).to_csv(
    remission_sensitivity_tables_dir
    / "remission_interval_sensitivity_confusion_matrix.csv"
)


## 10.3 REMISSION INTERVAL SHAP dependence threshold

The threshold is the midpoint of the strongest adjacent observed remission-duration interval where mean SHAP changes sign. Orange points are patient-level SHAP values; the shaded region is the observed transition interval.

In [ ]:
remission_sensitivity_pool = Pool(
    X_test_remission_sens_cb,
    cat_features=remission_sensitivity_cat_features
)
remission_sensitivity_shap_with_baseline = (
    remission_sensitivity_catboost_model.get_feature_importance(
        remission_sensitivity_pool,
        type=EFstrType.ShapValues
    )
)
remission_sensitivity_shap_values = np.asarray(
    remission_sensitivity_shap_with_baseline
)[:, :-1]
remission_feature_index = X_test_remission_sens_cb.columns.get_loc(
    "REMISSION INTERVAL"
)
remission_patient_shap = remission_sensitivity_shap_values[
    :, remission_feature_index
]
remission_patient_values = X_test_remission_sens_cb[
    "REMISSION INTERVAL"
].to_numpy(dtype=float)

remission_shap_dependence_df = (
    pd.DataFrame({
        "REMISSION INTERVAL": remission_patient_values,
        "SHAP_Value": remission_patient_shap
    })
    .groupby("REMISSION INTERVAL", as_index=False)["SHAP_Value"]
    .mean()
    .sort_values("REMISSION INTERVAL")
    .reset_index(drop=True)
)

remission_shap_x = remission_shap_dependence_df[
    "REMISSION INTERVAL"
].to_numpy(dtype=float)
remission_shap_y = remission_shap_dependence_df[
    "SHAP_Value"
].to_numpy(dtype=float)
remission_shap_crossings = np.where(
    ((remission_shap_y[:-1] <= 0) & (remission_shap_y[1:] > 0))
    | ((remission_shap_y[:-1] >= 0) & (remission_shap_y[1:] < 0))
)[0]

if len(remission_shap_crossings) > 0:
    remission_shap_changes = np.abs(np.diff(remission_shap_y))
    remission_shap_transition_index = remission_shap_crossings[
        np.argmax(remission_shap_changes[remission_shap_crossings])
    ]
else:
    remission_shap_transition_index = int(
        np.argmax(np.abs(np.diff(remission_shap_y)))
    )

remission_shap_transition_lower = float(
    remission_shap_x[remission_shap_transition_index]
)
remission_shap_transition_upper = float(
    remission_shap_x[remission_shap_transition_index + 1]
)
remission_interval_shap_threshold = float(
    (remission_shap_transition_lower + remission_shap_transition_upper) / 2
)
remission_shap_lower_effect = float(
    np.mean(remission_shap_y[:remission_shap_transition_index + 1])
)
remission_shap_upper_effect = float(
    np.mean(remission_shap_y[remission_shap_transition_index + 1:])
)
remission_shap_direction = (
    "Shorter remission interval associated with higher predicted relapse risk"
    if remission_shap_lower_effect > remission_shap_upper_effect
    else "Longer remission interval associated with higher predicted relapse risk"
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    remission_patient_values,
    remission_patient_shap,
    s=48,
    alpha=0.78,
    color="#E8753D",
    edgecolor="#C85A28",
    linewidth=0.8,
    label="Patient-level SHAP"
)
ax.axhline(0, color="grey", linewidth=0.8)
ax.axvspan(
    remission_shap_transition_lower,
    remission_shap_transition_upper,
    color="grey",
    alpha=0.14,
    label=(
        f"Observed transition {remission_shap_transition_lower:.1f}?"
        f"{remission_shap_transition_upper:.1f}"
    )
)
ax.axvline(
    remission_interval_shap_threshold,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Threshold ? {remission_interval_shap_threshold:.1f}"
)
ax.set_xlabel("REMISSION INTERVAL")
ax.set_ylabel("Sensitivity-model SHAP value")
ax.set_title("REMISSION INTERVAL SHAP Dependence")
ax.legend()
fig.tight_layout()
fig.savefig(
    remission_sensitivity_figures_dir
    / "remission_interval_shap_dependence.png",
    dpi=300,
    bbox_inches="tight"
)
plt.close(fig)


## 10.4 REMISSION INTERVAL EBM threshold

A separate additive EBM is used only to inspect the remission-duration risk curve. Its threshold is selected from the strongest zero-crossing transition, or the largest score change if no crossing is present.

In [ ]:
try:
    from interpret.glassbox import ExplainableBoostingClassifier
    REMISSION_EBM_AVAILABLE = True
except ImportError:
    REMISSION_EBM_AVAILABLE = False
    print(
        "The 'interpret' package is required for remission EBM analysis. "
        "Install it with: pip install interpret"
    )

if REMISSION_EBM_AVAILABLE:
    remission_ebm_sample_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train
    )
    remission_sensitivity_ebm_model = ExplainableBoostingClassifier(
        interactions=0,
        max_bins=64,
        outer_bags=8,
        learning_rate=0.03,
        max_rounds=3000,
        early_stopping_rounds=100,
        min_samples_leaf=4,
        n_jobs=1,
        random_state=42
    )
    remission_sensitivity_ebm_model.fit(
        X_train_remission_sens_encoded,
        y_train,
        sample_weight=remission_ebm_sample_weights
    )

    remission_ebm_global = remission_sensitivity_ebm_model.explain_global()
    remission_ebm_term_index = (
        remission_sensitivity_ebm_model.term_names_.index(
            "REMISSION INTERVAL"
        )
    )
    remission_ebm_term_data = remission_ebm_global.data(
        remission_ebm_term_index
    )
    remission_ebm_boundaries = np.asarray(
        remission_ebm_term_data["names"],
        dtype=float
    )
    remission_ebm_scores = np.asarray(
        remission_ebm_term_data["scores"],
        dtype=float
    )
    remission_ebm_x = (
        (remission_ebm_boundaries[:-1] + remission_ebm_boundaries[1:]) / 2
        if len(remission_ebm_boundaries) == len(remission_ebm_scores) + 1
        else remission_ebm_boundaries[:len(remission_ebm_scores)]
    )

    remission_ebm_changes = np.abs(np.diff(remission_ebm_scores))
    remission_ebm_crossings = np.where(
        ((remission_ebm_scores[:-1] <= 0) & (remission_ebm_scores[1:] > 0))
        | ((remission_ebm_scores[:-1] >= 0) & (remission_ebm_scores[1:] < 0))
    )[0]
    if len(remission_ebm_crossings) > 0:
        remission_ebm_transition_index = remission_ebm_crossings[
            np.argmax(remission_ebm_changes[remission_ebm_crossings])
        ]
    else:
        remission_ebm_transition_index = int(
            np.argmax(remission_ebm_changes)
        )

    remission_interval_ebm_threshold = float(
        remission_ebm_boundaries[remission_ebm_transition_index + 1]
    )
    remission_ebm_lower_effect = float(
        np.mean(remission_ebm_scores[:remission_ebm_transition_index + 1])
    )
    remission_ebm_upper_effect = float(
        np.mean(remission_ebm_scores[remission_ebm_transition_index + 1:])
    )
    remission_ebm_direction = (
        "Shorter remission interval associated with higher predicted relapse risk"
        if remission_ebm_lower_effect > remission_ebm_upper_effect
        else "Longer remission interval associated with higher predicted relapse risk"
    )

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(
        remission_ebm_x,
        remission_ebm_scores,
        color="#D95319",
        linewidth=2
    )
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.axvline(
        remission_interval_ebm_threshold,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Threshold ? {remission_interval_ebm_threshold:.1f}"
    )
    ax.set_xlabel("REMISSION INTERVAL")
    ax.set_ylabel("EBM contribution to relapse log-odds")
    ax.set_title("REMISSION INTERVAL EBM Risk Curve")
    ax.legend()
    fig.tight_layout()
    fig.savefig(
        remission_sensitivity_figures_dir
        / "remission_interval_ebm_curve.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.close(fig)
else:
    remission_sensitivity_ebm_model = None
    remission_interval_ebm_threshold = np.nan
    remission_ebm_direction = "Not assessable"


## 10.5?10.6 Compare thresholds and save outputs

Agreement reflects threshold proximity and whether SHAP and EBM identify the same shorter-versus-longer remission risk direction. These findings remain secondary sensitivity results.

In [ ]:
if pd.isna(remission_interval_ebm_threshold):
    remission_threshold_agreement = "Not assessable"
    remission_threshold_interpretation = (
        "EBM threshold unavailable; SHAP threshold only."
    )
else:
    remission_direction_agrees = (
        remission_shap_direction == remission_ebm_direction
    )
    remission_threshold_difference = abs(
        remission_interval_shap_threshold
        - remission_interval_ebm_threshold
    )
    remission_relative_difference = (
        remission_threshold_difference
        / max(abs(remission_interval_ebm_threshold), 1.0)
    )

    if remission_direction_agrees and remission_relative_difference <= 0.10:
        remission_threshold_agreement = "Agree"
    elif remission_direction_agrees and remission_relative_difference <= 0.20:
        remission_threshold_agreement = "Partially agree"
    else:
        remission_threshold_agreement = "Disagree"

    remission_threshold_interpretation = (
        f"SHAP threshold ? {remission_interval_shap_threshold:.1f}; "
        f"EBM threshold ? {remission_interval_ebm_threshold:.1f}. "
        f"SHAP: {remission_shap_direction}. "
        f"EBM: {remission_ebm_direction}."
    )

remission_interval_threshold_summary_df = pd.DataFrame([{
    "Feature": "REMISSION INTERVAL",
    "SHAP_Threshold": remission_interval_shap_threshold,
    "EBM_Threshold": remission_interval_ebm_threshold,
    "Agreement": remission_threshold_agreement,
    "Interpretation": remission_threshold_interpretation
}])
remission_interval_threshold_summary_df.to_csv(
    remission_sensitivity_tables_dir
    / "remission_interval_threshold_summary.csv",
    index=False
)

print("Sensitivity model metrics:")
print(remission_sensitivity_metrics)
print("Sensitivity confusion matrix:")
print(remission_sensitivity_confusion_matrix)
print("REMISSION INTERVAL SHAP threshold:")
print(remission_interval_shap_threshold)
print("REMISSION INTERVAL EBM threshold:")
print(remission_interval_ebm_threshold)
print("Threshold comparison:")
print(remission_interval_threshold_summary_df.to_string(index=False))


# 11. Final Findings Summary

This section consolidates completed notebook results for later abstract, poster, and presentation development. It reformats existing outputs only and does not retrain or alter any analytical model.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

final_tables_dir = Path("outputs") / "tables"
final_figures_dir = Path("outputs") / "figures"
final_tables_dir.mkdir(parents=True, exist_ok=True)
final_figures_dir.mkdir(parents=True, exist_ok=True)


def warn_missing(variable_name, section_name):
    warnings.warn(
        f"{section_name} skipped or incomplete because "
        f"'{variable_name}' is unavailable."
    )


def yes_no(value):
    return "Yes" if bool(value) else "No"


## 11.1 Model benchmark summary

In [ ]:
if "benchmark_results_df" in globals() and not benchmark_results_df.empty:
    final_model_benchmark_summary_df = benchmark_results_df[[
        "Model", "Precision", "Recall", "F1"
    ]].copy()
    final_model_benchmark_summary_df = final_model_benchmark_summary_df.rename(
        columns={"F1": "F1_Score"}
    )

    summary_best_model_name = (
        best_model_name
        if "best_model_name" in globals()
        else final_model_benchmark_summary_df.loc[
            final_model_benchmark_summary_df["F1_Score"].idxmax(),
            "Model"
        ]
    )
    final_model_benchmark_summary_df["Best_Model"] = (
        final_model_benchmark_summary_df["Model"]
        .eq(summary_best_model_name)
        .map(yes_no)
    )
    final_model_benchmark_summary_df = (
        final_model_benchmark_summary_df
        .sort_values("F1_Score", ascending=False)
        .reset_index(drop=True)
    )
    final_model_benchmark_summary_df.to_csv(
        final_tables_dir / "final_model_benchmark_summary.csv",
        index=False
    )

    summary_best_row = final_model_benchmark_summary_df.iloc[0]
    print(f"Best model name: {summary_best_row['Model']}")
    print(f"Best model precision: {summary_best_row['Precision']:.4f}")
    print(f"Best model recall: {summary_best_row['Recall']:.4f}")
    print(f"Best model F1-score: {summary_best_row['F1_Score']:.4f}")
else:
    warn_missing("benchmark_results_df", "Model benchmark summary")
    final_model_benchmark_summary_df = pd.DataFrame(columns=[
        "Model", "Precision", "Recall", "F1_Score", "Best_Model"
    ])


## 11.2 Final selected predictors summary

Encoded LR/RF/MLP columns are mapped back to their original clinical source feature so selection methods can be compared meaningfully.

In [ ]:
selection_variables = {
    "LR": "selected_features_lr",
    "RF": "selected_features_rf",
    "MLP": "selected_features_mlp",
    "CatBoost": "selected_features_catboost"
}
missing_selection_variables = [
    variable_name
    for variable_name in selection_variables.values()
    if variable_name not in globals()
]

if not missing_selection_variables:
    def selection_source_features(features):
        source_features = []
        for feature in features:
            try:
                source_feature = encoded_to_original_feature(feature)
            except (KeyError, NameError):
                source_feature = feature
            if source_feature not in source_features:
                source_features.append(source_feature)
        return source_features

    selected_sources_lr = selection_source_features(selected_features_lr)
    selected_sources_rf = selection_source_features(selected_features_rf)
    selected_sources_mlp = selection_source_features(selected_features_mlp)
    selected_sources_catboost = selection_source_features(
        selected_features_catboost
    )

    all_selected_sources = []
    for feature_list in (
        selected_sources_lr,
        selected_sources_rf,
        selected_sources_mlp,
        selected_sources_catboost
    ):
        for feature in feature_list:
            if feature not in all_selected_sources:
                all_selected_sources.append(feature)

    final_selected_predictors_summary_df = pd.DataFrame({
        "Feature": all_selected_sources
    })
    final_selected_predictors_summary_df["Selected_by_LR"] = (
        final_selected_predictors_summary_df["Feature"]
        .isin(selected_sources_lr)
        .map(yes_no)
    )
    final_selected_predictors_summary_df["Selected_by_RF"] = (
        final_selected_predictors_summary_df["Feature"]
        .isin(selected_sources_rf)
        .map(yes_no)
    )
    final_selected_predictors_summary_df["Selected_by_MLP_Consensus"] = (
        final_selected_predictors_summary_df["Feature"]
        .isin(selected_sources_mlp)
        .map(yes_no)
    )
    final_selected_predictors_summary_df["Selected_by_CatBoost"] = (
        final_selected_predictors_summary_df["Feature"]
        .isin(selected_sources_catboost)
        .map(yes_no)
    )
    indicator_columns = [
        "Selected_by_LR",
        "Selected_by_RF",
        "Selected_by_MLP_Consensus",
        "Selected_by_CatBoost"
    ]
    final_selected_predictors_summary_df["Selection_Count"] = (
        final_selected_predictors_summary_df[indicator_columns]
        .eq("Yes")
        .sum(axis=1)
    )
    final_selected_predictors_summary_df = (
        final_selected_predictors_summary_df
        .sort_values(["Selection_Count", "Feature"], ascending=[False, True])
        .reset_index(drop=True)
    )
    final_selected_predictors_summary_df.to_csv(
        final_tables_dir / "final_selected_predictors_summary.csv",
        index=False
    )

    print("LR-selected features:", selected_features_lr)
    print("RF-selected features:", selected_features_rf)
    print("MLP consensus features:", selected_features_mlp)
    print("CatBoost-selected features:", selected_features_catboost)
else:
    for variable_name in missing_selection_variables:
        warn_missing(variable_name, "Selected predictors summary")
    final_selected_predictors_summary_df = pd.DataFrame(columns=[
        "Feature", "Selected_by_LR", "Selected_by_RF",
        "Selected_by_MLP_Consensus", "Selected_by_CatBoost",
        "Selection_Count"
    ])


## 11.3 Standard and Conditional SHAP summary

In [ ]:
if "shap_comparison_df" in globals() and not shap_comparison_df.empty:
    final_shap_conditional_summary_df = shap_comparison_df.rename(columns={
        "MeanAbsSHAP": "Standard_SHAP_Importance",
        "MeanAbsConditionalSHAP": "Conditional_SHAP_Importance"
    })[[
        "Feature",
        "Standard_SHAP_Rank",
        "Conditional_SHAP_Rank",
        "Rank_Change",
        "Standard_SHAP_Importance",
        "Conditional_SHAP_Importance",
        "Interpretation"
    ]].copy()
    final_shap_conditional_summary_df = (
        final_shap_conditional_summary_df
        .sort_values("Standard_SHAP_Rank")
        .reset_index(drop=True)
    )
    final_shap_conditional_summary_df.to_csv(
        final_tables_dir / "final_shap_conditional_summary.csv",
        index=False
    )

    stable_predictors = final_shap_conditional_summary_df.loc[
        final_shap_conditional_summary_df["Interpretation"] == "Stable",
        "Feature"
    ].tolist()
    changed_importance_features = final_shap_conditional_summary_df.loc[
        final_shap_conditional_summary_df["Interpretation"] != "Stable",
        "Feature"
    ].tolist()
    top_standard_features = final_shap_conditional_summary_df.nsmallest(
        5, "Standard_SHAP_Rank"
    )["Feature"].tolist()
    top_conditional_features = final_shap_conditional_summary_df.nsmallest(
        5, "Conditional_SHAP_Rank"
    )["Feature"].tolist()

    print("Top Standard SHAP features:", top_standard_features)
    print("Top Conditional SHAP features:", top_conditional_features)
    print("Stable predictors:", stable_predictors)
    print("Features with changed importance:", changed_importance_features)
else:
    warn_missing("shap_comparison_df", "SHAP comparison summary")
    final_shap_conditional_summary_df = pd.DataFrame(columns=[
        "Feature", "Standard_SHAP_Rank", "Conditional_SHAP_Rank",
        "Rank_Change", "Standard_SHAP_Importance",
        "Conditional_SHAP_Importance", "Interpretation"
    ])
    stable_predictors = []
    changed_importance_features = []


## 11.4 EBM threshold summary

Thresholds remain exploratory model-derived signals, not definitive clinical cut-offs.

In [ ]:
if (
    "ebm_threshold_summary_df" in globals()
    and "ebm_vs_shap_threshold_df" in globals()
    and not ebm_threshold_summary_df.empty
):
    ebm_comparison_columns = [
        "Feature", "EBM_Threshold", "SHAP_Dependence_Threshold",
        "Agreement", "Interpretation"
    ]
    available_columns = [
        column for column in ebm_comparison_columns
        if column in ebm_vs_shap_threshold_df.columns
    ]
    final_ebm_threshold_summary_df = (
        ebm_vs_shap_threshold_df[available_columns].copy()
    )
    for missing_column in (
        set(ebm_comparison_columns)
        - set(final_ebm_threshold_summary_df.columns)
    ):
        final_ebm_threshold_summary_df[missing_column] = np.nan
    final_ebm_threshold_summary_df = final_ebm_threshold_summary_df[
        ebm_comparison_columns
    ]
    final_ebm_threshold_summary_df["Interpretation"] = (
        final_ebm_threshold_summary_df["Interpretation"]
        .astype(str)
        .str.replace(" ? ", " approximately ", regex=False)
    )
    final_ebm_threshold_summary_df.to_csv(
        final_tables_dir / "final_ebm_threshold_summary.csv",
        index=False
    )

    ebm_agreement_features = final_ebm_threshold_summary_df.loc[
        final_ebm_threshold_summary_df["Agreement"].isin([
            "Agree", "Partially agree"
        ]),
        "Feature"
    ].tolist()
    ebm_unclear_features = final_ebm_threshold_summary_df.loc[
        ~final_ebm_threshold_summary_df["Agreement"].isin([
            "Agree", "Partially agree"
        ]),
        "Feature"
    ].tolist()

    print("Main EBM threshold findings:")
    print(final_ebm_threshold_summary_df.to_string(index=False))
    print("EBM/SHAP agreement features:", ebm_agreement_features)
    print("Unclear threshold agreement:", ebm_unclear_features)
else:
    if "ebm_threshold_summary_df" not in globals():
        warn_missing("ebm_threshold_summary_df", "EBM threshold summary")
    if "ebm_vs_shap_threshold_df" not in globals():
        warn_missing("ebm_vs_shap_threshold_df", "EBM threshold summary")
    final_ebm_threshold_summary_df = pd.DataFrame(columns=[
        "Feature", "EBM_Threshold", "SHAP_Dependence_Threshold",
        "Agreement", "Interpretation"
    ])
    ebm_agreement_features = []
    ebm_unclear_features = []


## 11.5 DML exploratory summary

DML remains an exploratory adjusted-association layer and is not a prediction benchmark or definitive causal analysis.

In [ ]:
if "dml_results_df" in globals() and not dml_results_df.empty:
    final_dml_summary_df = dml_results_df[[
        "Feature", "Estimated_Effect", "CI_Lower", "CI_Upper",
        "P_Value", "Direction", "Interpretation"
    ]].copy()

    def dml_signal_category(direction):
        if direction == "Positive adjusted signal with relapse":
            return "Positive adjusted signal"
        if direction == "Negative/protective adjusted signal with relapse":
            return "Negative/protective adjusted signal"
        return "No clear adjusted signal"

    final_dml_summary_df["Signal_Category"] = (
        final_dml_summary_df["Direction"].apply(dml_signal_category)
    )
    final_dml_summary_df.to_csv(
        final_tables_dir / "final_dml_summary.csv",
        index=False
    )

    final_positive_dml = final_dml_summary_df.loc[
        final_dml_summary_df["Signal_Category"]
        == "Positive adjusted signal",
        "Feature"
    ].tolist()
    final_negative_dml = final_dml_summary_df.loc[
        final_dml_summary_df["Signal_Category"]
        == "Negative/protective adjusted signal",
        "Feature"
    ].tolist()
    final_unclear_dml = final_dml_summary_df.loc[
        final_dml_summary_df["Signal_Category"]
        == "No clear adjusted signal",
        "Feature"
    ].tolist()

    print("Positive adjusted signals:", final_positive_dml)
    print("Negative/protective adjusted signals:", final_negative_dml)
    print("Features with no clear adjusted signal:", final_unclear_dml)
else:
    warn_missing("dml_results_df", "DML summary")
    final_dml_summary_df = pd.DataFrame(columns=[
        "Feature", "Estimated_Effect", "CI_Lower", "CI_Upper",
        "P_Value", "Direction", "Interpretation", "Signal_Category"
    ])
    final_positive_dml = []
    final_negative_dml = []
    final_unclear_dml = []


## 11.6 REMISSION INTERVAL sensitivity summary

In [ ]:
if (
    "remission_interval_threshold_summary_df" in globals()
    and not remission_interval_threshold_summary_df.empty
):
    final_remission_interval_sensitivity_summary_df = (
        remission_interval_threshold_summary_df.loc[
            remission_interval_threshold_summary_df["Feature"]
            == "REMISSION INTERVAL",
            [
                "Feature", "SHAP_Threshold", "EBM_Threshold",
                "Agreement", "Interpretation"
            ]
        ].copy()
    )
    final_remission_interval_sensitivity_summary_df["Interpretation"] = (
        final_remission_interval_sensitivity_summary_df["Interpretation"]
        .astype(str)
        .str.replace(" ? ", " approximately ", regex=False)
    )
    final_remission_interval_sensitivity_summary_df[
        "Sensitivity_Only_Note"
    ] = (
        "Secondary sensitivity analysis only; REMISSION INTERVAL was excluded "
        "from the primary predictive model due to possible outcome-timing leakage."
    )
    final_remission_interval_sensitivity_summary_df.to_csv(
        final_tables_dir
        / "final_remission_interval_sensitivity_summary.csv",
        index=False
    )

    remission_summary_row = (
        final_remission_interval_sensitivity_summary_df.iloc[0]
    )
    print("REMISSION INTERVAL SHAP threshold:", remission_summary_row["SHAP_Threshold"])
    print("REMISSION INTERVAL EBM threshold:", remission_summary_row["EBM_Threshold"])
    print("Clinical interpretation:", remission_summary_row["Interpretation"])
    print("Sensitivity-only note:", remission_summary_row["Sensitivity_Only_Note"])
else:
    warn_missing(
        "remission_interval_threshold_summary_df",
        "REMISSION INTERVAL sensitivity summary"
    )
    final_remission_interval_sensitivity_summary_df = pd.DataFrame(columns=[
        "Feature", "SHAP_Threshold", "EBM_Threshold", "Agreement",
        "Interpretation", "Sensitivity_Only_Note"
    ])


## 11.7 Final competition-ready bullet summary

In [ ]:
print("\n=== FINAL COMPETITION-READY FINDINGS ===")

if not final_model_benchmark_summary_df.empty:
    best_summary = final_model_benchmark_summary_df.iloc[0]
    print(
        "1. Best predictive model: "
        f"{best_summary['Model']} "
        f"(precision {best_summary['Precision']:.3f}, "
        f"recall {best_summary['Recall']:.3f}, "
        f"F1 {best_summary['F1_Score']:.3f})."
    )
else:
    print("1. Best predictive model: unavailable.")

print(
    "2. Stable relapse-related predictors: "
    + (", ".join(stable_predictors) if stable_predictors else "not identified")
    + "."
)
print(
    "3. Explanation robustness findings: "
    + (
        "Standard and Conditional SHAP rankings were stable for the leading predictors."
        if stable_predictors and not changed_importance_features
        else "Some predictor rankings changed after correlation-aware explanation."
    )
)

if not final_ebm_threshold_summary_df.empty:
    main_numeric_thresholds = final_ebm_threshold_summary_df.loc[
        final_ebm_threshold_summary_df["Feature"] == "LOWEST HB"
    ]
    if not main_numeric_thresholds.empty:
        threshold_row = main_numeric_thresholds.iloc[0]
        print(
            "4. Clinically readable threshold findings: LOWEST HB showed "
            f"an exploratory SHAP threshold near "
            f"{threshold_row['SHAP_Dependence_Threshold']:.2f} and EBM "
            f"threshold near {threshold_row['EBM_Threshold']:.2f} "
            f"({threshold_row['Agreement']})."
        )
    else:
        print("4. Clinically readable threshold findings: see final EBM summary table.")
else:
    print("4. Clinically readable threshold findings: unavailable.")

print(
    "5. Exploratory DML findings: positive signals - "
    + (", ".join(final_positive_dml) if final_positive_dml else "none")
    + "; negative/protective signals - "
    + (", ".join(final_negative_dml) if final_negative_dml else "none")
    + "."
)

if not final_remission_interval_sensitivity_summary_df.empty:
    remission_row = final_remission_interval_sensitivity_summary_df.iloc[0]
    print(
        "6. REMISSION INTERVAL sensitivity finding: shorter remission "
        f"duration showed exploratory transitions near SHAP "
        f"{remission_row['SHAP_Threshold']:.1f} and EBM "
        f"{remission_row['EBM_Threshold']:.1f}; sensitivity analysis only."
    )
else:
    print("6. REMISSION INTERVAL sensitivity finding: unavailable.")

print(
    "7. Main methodological caution: SHAP explains model behavior, not causality; "
    "Conditional SHAP is correlation-aware, not causal proof; EBM thresholds "
    "are exploratory and not definitive clinical cut-offs; DML explores "
    "possible causal signals after adjustment for observed covariates, not "
    "definitive causality; REMISSION INTERVAL was excluded from the primary "
    "model and used only in secondary sensitivity analysis."
)


# 12. Abstract Writing ? Later Step

The final abstract remains intentionally unwritten.